<div style='border-left:6px solid #17365D;padding:10px 16px;margin-bottom:10px;background:#F2F6FB'>
<h1 style='margin:0;color:#17365D'>🚀 Automatizador de determinación mensual de IVA e IT</h1>
<p style='margin:6px 0 0;color:#404040'>UNIVALLE &middot; Cruce SIAT (NetValle) &times; Mayor SAP</p>
</div>

**Instrucciones:**

1. Abre este archivo en Google Colab.
2. Presiona el botón ▶️ de la única celda de abajo (**GENERAR DETERMINACIÓN IVA–IT**).
3. Cuando se abra el selector de archivos, elige **juntos** los 2 archivos `.xlsx`: el **CRUCE OFICIAL** (contiene la pestaña `SIAT`) y el **mayor SAP** (contiene la pestaña `SAP Document Export`).
4. Espera a que la barra de progreso llegue a 100%. El Excel se descargará automáticamente; si el navegador bloquea la descarga, aparecerá un enlace manual.

No necesitas ejecutar ninguna otra celda, ni seleccionar "Ejecutar todas", ni desplegar el código.

> El mayor SAP debe contener las cuentas `610501001`, `210106001`, `110205003` y cuentas de ingreso (que comienzan con `4`) con movimiento en el periodo.


In [ ]:
#@title 🚀 GENERAR DETERMINACIÓN IVA–IT { display-mode: "form" }
# Motor, maestros y panel reunidos en una sola celda oculta.
from __future__ import annotations

import re
from collections import defaultdict
from datetime import date, datetime
from pathlib import Path
from typing import Any

import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


ACCOUNT_IT = "610501001"
ACCOUNT_VAT = "210106001"
ACCOUNT_DISCOUNT_VAT = "110205003"

SITE_BY_SEGMENT = {
    "S100": "Cochabamba",
    "S200": "La Paz",
    "S300": "Sucre",
    "S400": "Trinidad",
    "S500": "Santa Cruz",
}
SITE_ORDER = ["Cochabamba", "La Paz", "Santa Cruz", "Sucre", "Trinidad"]

# Las reglas pueden ampliarse sin cambiar el motor. Una cuenta nueva no se
# incorpora silenciosamente al IT: queda marcada para revisión.
ACCOUNT_RULES = {
    "410201001": {"name": "Ingresos por colegiatura", "group": "Ingresos académicos", "it": False},
    "410201010": {"name": "Descuentos en ventas", "group": "Descuentos", "it": False},
    "410203001": {"name": "Cursos de invierno", "group": "Ingresos académicos", "it": False},
    "410205002": {"name": "Cursos de maestría", "group": "Ingresos académicos", "it": False},
    "410205003": {"name": "Cursos de diplomado", "group": "Ingresos académicos", "it": False},
    "410207001": {"name": "Formularios y valores - IAFVV", "group": "IAFVV", "it": True},
    "410207002": {"name": "Interés legal - IAILC", "group": "IAILC", "it": True},
    "410208002": {"name": "Cursos de formación continua", "group": "Ingresos académicos", "it": False},
    "420101001": {"name": "Ingresos por alquileres", "group": "Otros ingresos", "it": True},
    "420101003": {"name": "Ingresos por intereses", "group": "Otros no alcanzados", "it": False},
    "420101006": {"name": "Ingresos Clínica odontológica", "group": "Clínica odontológica", "it": True},
    "420101008": {"name": "Ingresos académicos editorial", "group": "Venta de libros", "it": False},
    "420101009": {"name": "Ingresos por venta de souvenirs", "group": "Souvenirs", "it": True},
    "420101010": {"name": "Varios otros", "group": "Otros ingresos", "it": True},
    "420101011": {"name": "Otros ingresos por descuento en compras", "group": "Otros no alcanzados", "it": False},
    "420201002": {"name": "Corrección costo inventario", "group": "Ajustes contables", "it": False},
    "420301001": {"name": "Diferencia por redondeo", "group": "Ajustes contables", "it": False},
    "420301002": {"name": "Diferencia de cambio", "group": "Ajustes contables", "it": False},
    "420301004": {"name": "Diferencia por redondeo inventario", "group": "Ajustes contables", "it": False},
}

COMPOSITION_GROUPS = [
    "Clínica odontológica",
    "Souvenirs",
    "Otros ingresos",
    "IAILC",
    "IAFVV",
]

SIAT_REQUIRED = [
    "SEDE SEGÚN NETVALLE",
    "CAJA / PUNTO SEGÚN NETVALLE",
    "FECHA DE LA FACTURA",
    "Nº DE LA FACTURA",
    "CODIGO DE AUTORIZACIÓN",
    "IMPORTE TOTAL DE LA VENTA",
    "VENTAS GRAVADAS A TASA CERO",
    "DESCUENTOS BONIFICACIONES Y REBAJAS SUJETAS AL IVA",
    "IMPORTE BASE PARA DEBITO FISCAL",
    "DEBITO FISCAL",
    "ESTADO",
]

SAP_REQUIRED = [
    "Cuenta de mayor",
    "Nom.larg.cta.mayor",
    "Asiento contable",
    "Fe.contab.",
    "Clave contab.",
    "Impte.moneda socied.",
    "Centro de beneficio",
    "Segmento",
    "Txt.pos.reg.diario",
    "AC creado por",
    "Está anulada",
    "Está anulando",
]


def clean_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value).strip()


def normalize_account(value: Any) -> str:
    return re.sub(r"\.0$", "", clean_text(value))


def normalize_invoice(value: Any) -> str:
    digits = re.sub(r"\D", "", clean_text(value))
    return digits.lstrip("0") or ("0" if digits else "")


def extract_invoice(value: Any) -> str:
    text = clean_text(value)
    match = re.search(r"(?i)(?:^|\s)F\s*[.\-:]?\s*(\d+)", text)
    return normalize_invoice(match.group(1)) if match else ""


def normalize_date(value: Any) -> str:
    if isinstance(value, datetime):
        return value.date().isoformat()
    if isinstance(value, date):
        return value.isoformat()
    parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")
    if pd.isna(parsed):
        return clean_text(value)
    return parsed.date().isoformat()


def require_columns(frame: pd.DataFrame, required: list[str], label: str) -> None:
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"{label}: faltan columnas obligatorias: {', '.join(missing)}")


def period_label(dates: list[str]) -> tuple[str, str, str]:
    parsed = sorted(pd.to_datetime([item for item in dates if item], errors="coerce").dropna())
    if not parsed:
        raise ValueError("No se pudo determinar el periodo de los archivos.")
    first, last = parsed[0], parsed[-1]
    months = {(item.year, item.month) for item in parsed}
    if len(months) != 1:
        raise ValueError("Los archivos contienen más de un mes. Exporte un solo periodo mensual.")
    months_es = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre",
    ]
    label = f"{months_es[first.month - 1].capitalize()} {first.year}"
    return label, first.strftime("%d/%m/%Y"), last.strftime("%d/%m/%Y")


def load_inputs(siat_path: str | Path, sap_path: str | Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    try:
        siat = pd.read_excel(siat_path, sheet_name="SIAT")
    except ValueError as exc:
        raise ValueError("El libro de cruce no contiene la pestaña 'SIAT'.") from exc
    try:
        sap = pd.read_excel(sap_path, sheet_name="SAP Document Export")
    except ValueError as exc:
        raise ValueError("El mayor SAP no contiene la pestaña 'SAP Document Export'.") from exc
    require_columns(siat, SIAT_REQUIRED, "Libro SIAT")
    require_columns(sap, SAP_REQUIRED, "Mayor SAP")
    return siat, sap


def process_data(
    siat: pd.DataFrame,
    sap: pd.DataFrame,
    cebe_map: dict[str, dict[str, Any]] | None = None,
) -> dict[str, Any]:
    cebe_map = cebe_map or {}
    siat = siat.copy()
    sap = sap.copy()

    sap["_account"] = sap["Cuenta de mayor"].map(normalize_account)
    sap["_amount"] = pd.to_numeric(sap["Impte.moneda socied."], errors="coerce").fillna(0.0)
    sap["_date"] = sap["Fe.contab."].map(normalize_date)
    sap["_document"] = sap["Asiento contable"].map(clean_text)
    sap["_center"] = sap["Centro de beneficio"].map(clean_text)
    sap["_segment"] = sap["Segmento"].map(lambda value: clean_text(value).upper())
    sap["_creator"] = sap["AC creado por"].map(lambda value: clean_text(value).upper())

    siat["_invoice"] = siat["Nº DE LA FACTURA"].map(normalize_invoice)
    siat["_date"] = siat["FECHA DE LA FACTURA"].map(normalize_date)
    siat["_state"] = siat["ESTADO"].map(lambda value: clean_text(value).upper())
    siat["_box"] = siat["CAJA / PUNTO SEGÚN NETVALLE"].map(lambda value: clean_text(value).upper())
    siat["_site"] = siat["SEDE SEGÚN NETVALLE"].map(lambda value: clean_text(value).upper())

    sap_label, sap_from, sap_to = period_label(sap["_date"].tolist())
    siat_label, siat_from, siat_to = period_label(siat["_date"].tolist())
    if sap_label != siat_label:
        raise ValueError(f"Los periodos no coinciden: SIAT {siat_label}, SAP {sap_label}.")

    present_accounts = set(sap["_account"])
    required_accounts = {ACCOUNT_IT, ACCOUNT_VAT, ACCOUNT_DISCOUNT_VAT}
    missing_tax_accounts = sorted(required_accounts - present_accounts)
    if missing_tax_accounts:
        raise ValueError("El mayor SAP no contiene las cuentas: " + ", ".join(missing_tax_accounts))

    unknown_income = sorted(
        account for account in present_accounts
        if account.startswith("4") and account not in ACCOUNT_RULES
    )

    # Índices del SIAT por factura y fecha. Los anulados se mantienen separados.
    valid_indices: dict[tuple[str, str], list[int]] = defaultdict(list)
    annulled_indices: dict[tuple[str, str], list[int]] = defaultdict(list)
    for index, row in siat.iterrows():
        key = (row["_invoice"], row["_date"])
        if row["_state"] == "VALIDA":
            valid_indices[key].append(index)
        elif row["_state"] == "ANULADA":
            annulled_indices[key].append(index)

    # IT por factura: se agrupa respetando el signo. Un reverso neto cero no se asigna.
    it_lines = sap[sap["_account"] == ACCOUNT_IT].copy()
    it_lines["_invoice"] = it_lines["Txt.pos.reg.diario"].map(extract_invoice)
    unparsed_lines = it_lines[it_lines["_invoice"] == ""]
    groups: dict[tuple[str, str], dict[str, Any]] = defaultdict(
        lambda: {"amount": 0.0, "lines": 0, "creators": set(), "segments": set(), "texts": []}
    )
    for _, row in it_lines[it_lines["_invoice"] != ""].iterrows():
        key = (row["_invoice"], row["_date"])
        item = groups[key]
        item["amount"] += float(row["_amount"])
        item["lines"] += 1
        if row["_creator"]:
            item["creators"].add(row["_creator"])
        if row["_segment"]:
            item["segments"].add(row["_segment"])
        if len(item["texts"]) < 3:
            item["texts"].append(clean_text(row["Txt.pos.reg.diario"]))

    it_per_row = pd.Series(0.0, index=siat.index)
    methods: dict[str, int] = defaultdict(int)
    zero_groups = 0
    exceptions: list[dict[str, Any]] = []
    for (invoice, invoice_date), item in groups.items():
        amount = item["amount"]
        if abs(amount) < 0.005:
            zero_groups += 1
            continue
        candidates = list(valid_indices.get((invoice, invoice_date), []))
        method = "Factura + fecha"
        if len(candidates) > 1:
            by_box = [index for index in candidates if siat.at[index, "_box"] in item["creators"]]
            if len(by_box) == 1:
                candidates = by_box
                method = "Factura + fecha + caja"
        if len(candidates) > 1:
            expected_sites = {
                SITE_BY_SEGMENT.get(segment, segment).upper()
                for segment in item["segments"] if segment
            }
            by_site = [index for index in candidates if siat.at[index, "_site"] in expected_sites]
            if len(by_site) == 1:
                candidates = by_site
                method = "Factura + fecha + sede"
        if len(candidates) == 1:
            it_per_row.at[candidates[0]] += amount
            methods[method] += 1
            continue

        annulled = annulled_indices.get((invoice, invoice_date), [])
        if annulled:
            reason = "Factura anulada en SIAT con IT SAP neto distinto de cero"
        elif not candidates:
            reason = "Factura SAP sin coincidencia válida en SIAT"
        else:
            reason = "Factura duplicada sin desempate suficiente"
        exceptions.append({
            "tipo": reason,
            "factura": invoice,
            "fecha": invoice_date,
            "it_sap": amount,
            "candidatos_validos": len(candidates),
            "candidatos_anulados": len(annulled),
            "cajas_sap": ", ".join(sorted(item["creators"])),
            "segmentos_sap": ", ".join(sorted(item["segments"])),
            "glosa": " | ".join(item["texts"]),
        })

    for _, row in unparsed_lines.iterrows():
        exceptions.append({
            "tipo": "Glosa de IT sin número de factura reconocible",
            "factura": "",
            "fecha": row["_date"],
            "it_sap": float(row["_amount"]),
            "candidatos_validos": 0,
            "candidatos_anulados": 0,
            "cajas_sap": row["_creator"],
            "segmentos_sap": row["_segment"],
            "glosa": clean_text(row["Txt.pos.reg.diario"]),
        })

    it_source_total = float(it_lines["_amount"].sum())
    it_assigned_total = float(it_per_row.sum())
    assignment_difference = it_source_total - it_assigned_total

    # Composición del IT por naturaleza: cuenta de ingreso del mismo asiento y CeBe.
    it_by_doc_center: dict[tuple[str, str], float] = defaultdict(float)
    segment_by_doc_center: dict[tuple[str, str], str] = {}
    for _, row in it_lines.iterrows():
        key = (row["_document"], row["_center"])
        it_by_doc_center[key] += float(row["_amount"])
        segment_by_doc_center[key] = row["_segment"]

    income = sap[sap["_account"].str.startswith("4")].copy()
    income_by_doc_center: dict[tuple[str, str], dict[str, float]] = defaultdict(lambda: defaultdict(float))
    income_by_document: dict[str, dict[str, float]] = defaultdict(lambda: defaultdict(float))
    for _, row in income.iterrows():
        income_by_doc_center[(row["_document"], row["_center"])][row["_account"]] += float(row["_amount"])
        income_by_document[row["_document"]][row["_account"]] += float(row["_amount"])

    group_site_it: dict[tuple[str, str], float] = defaultdict(float)
    link_methods: dict[str, int] = defaultdict(int)
    for key, it_amount in it_by_doc_center.items():
        if abs(it_amount) < 0.005:
            continue
        document, center = key
        candidates = [
            (account, amount)
            for account, amount in income_by_doc_center.get(key, {}).items()
            if ACCOUNT_RULES.get(account, {}).get("it") and abs(amount) >= 0.005
        ]
        method = "Documento + CeBe"
        if not candidates:
            candidates = [
                (account, amount)
                for account, amount in income_by_document.get(document, {}).items()
                if ACCOUNT_RULES.get(account, {}).get("it") and abs(amount) >= 0.005
            ]
            method = "Solo documento"
        segment = segment_by_doc_center.get(key, "")
        site = SITE_BY_SEGMENT.get(segment, segment or "Sin sede")
        if not candidates:
            group_site_it[("Sin cuenta vinculada", site)] += it_amount
            link_methods["Sin cuenta vinculada"] += 1
            exceptions.append({
                "tipo": "IT sin cuenta de ingreso alcanzada en el mismo asiento",
                "factura": "",
                "fecha": "",
                "it_sap": it_amount,
                "candidatos_validos": 0,
                "candidatos_anulados": 0,
                "cajas_sap": "",
                "segmentos_sap": segment,
                "glosa": f"Asiento {document}; CeBe {center}",
            })
            continue
        denominator = sum(abs(amount) for _, amount in candidates)
        for account, amount in candidates:
            group = ACCOUNT_RULES[account]["group"]
            group_site_it[(group, site)] += it_amount * abs(amount) / denominator
        link_methods[method] += 1

    # Detalle del IT por CeBe, conservando débitos, créditos/reversos y neto.
    center_detail: dict[str, dict[str, Any]] = defaultdict(
        lambda: {"moves": 0, "debits": 0.0, "credits": 0.0, "net": 0.0, "segment": ""}
    )
    for _, row in it_lines.iterrows():
        center = row["_center"]
        amount = float(row["_amount"])
        item = center_detail[center]
        item["moves"] += 1
        item["net"] += amount
        item["segment"] = row["_segment"]
        if amount >= 0:
            item["debits"] += amount
        else:
            item["credits"] += amount

    center_rows = []
    for center, item in center_detail.items():
        meta = cebe_map.get(center, {})
        center_rows.append({
            "center": center,
            "site": SITE_BY_SEGMENT.get(item["segment"], item["segment"] or "Sin sede"),
            "segment": item["segment"],
            "name": meta.get("denominacion", "Sin denominación en maestro"),
            "area": {"A001": "Administrativas", "A002": "Carreras", "A003": "Laboratorios"}.get(
                meta.get("area_codigo", ""), "Sin área"
            ),
            "moves": item["moves"],
            "debits": item["debits"],
            "credits": item["credits"],
            "net": item["net"],
            "base": item["net"] / 0.03,
            "blocked": "Sí" if meta.get("bloqueado") else "No",
        })
    center_rows.sort(key=lambda item: item["net"], reverse=True)

    numeric_siat_columns = [
        "IMPORTE TOTAL DE LA VENTA", "IMPORTE ICE", "IMPORTE IEHD", "IMPORTE IPJ",
        "TASAS", "OTROS NO SUJETOS AL IVA", "EXPORTACIONES Y OPERACIONES EXENTAS",
        "VENTAS GRAVADAS A TASA CERO", "SUBTOTAL",
        "DESCUENTOS BONIFICACIONES Y REBAJAS SUJETAS AL IVA", "IMPORTE GIFT CARD",
        "IMPORTE BASE PARA DEBITO FISCAL", "DEBITO FISCAL",
    ]
    valid_siat = siat[siat["_state"] == "VALIDA"]
    siat_totals = {
        column: float(pd.to_numeric(valid_siat[column], errors="coerce").fillna(0).sum())
        for column in numeric_siat_columns if column in siat.columns
    }
    siat_counts = {
        "all": len(siat),
        "valid": int((siat["_state"] == "VALIDA").sum()),
        "annulled": int((siat["_state"] == "ANULADA").sum()),
    }

    sap_vat_gross = -float(sap.loc[sap["_account"] == ACCOUNT_VAT, "_amount"].sum())
    sap_discount_credit = float(sap.loc[sap["_account"] == ACCOUNT_DISCOUNT_VAT, "_amount"].sum())
    sap_vat_net = sap_vat_gross - sap_discount_credit

    composition = []
    all_groups = COMPOSITION_GROUPS + (["Sin cuenta vinculada"] if any(
        group == "Sin cuenta vinculada" for group, _ in group_site_it
    ) else [])
    for group in all_groups:
        by_site = {site: group_site_it[(group, site)] for site in SITE_ORDER}
        other_sites = sorted({site for g, site in group_site_it if g == group and site not in SITE_ORDER})
        for site in other_sites:
            by_site[site] = group_site_it[(group, site)]
        it_amount = sum(by_site.values())
        composition.append({"group": group, "by_site": by_site, "it": it_amount, "base": it_amount / 0.03})

    income_accounts = []
    for account, frame in income.groupby("_account", sort=True):
        rule = ACCOUNT_RULES.get(account, {})
        amount = float(frame["_amount"].sum())
        income_accounts.append({
            "account": account,
            "name": clean_text(frame["Nom.larg.cta.mayor"].dropna().iloc[0]) if frame["Nom.larg.cta.mayor"].notna().any() else rule.get("name", ""),
            "moves": len(frame),
            "income_net": -amount,
            "group": rule.get("group", "REVISAR"),
            "it_rule": "Sí" if rule.get("it") else ("No" if account in ACCOUNT_RULES else "Revisar"),
        })

    return {
        "period": {"label": sap_label, "from": sap_from, "to": sap_to},
        "siat": siat,
        "sap": sap,
        "it_per_row": it_per_row,
        "siat_totals": siat_totals,
        "siat_counts": siat_counts,
        "it_source_total": it_source_total,
        "it_assigned_total": it_assigned_total,
        "assignment_difference": assignment_difference,
        "it_lines": len(it_lines),
        "it_groups": len(groups),
        "zero_groups": zero_groups,
        "assignment_methods": dict(methods),
        "composition": composition,
        "link_methods": dict(link_methods),
        "center_rows": center_rows,
        "income_accounts": income_accounts,
        "unknown_income_accounts": unknown_income,
        "sap_vat_gross": sap_vat_gross,
        "sap_discount_credit": sap_discount_credit,
        "sap_vat_net": sap_vat_net,
        "exceptions": exceptions,
        "input_rows": {"siat": len(siat), "sap": len(sap)},
    }


NAVY = "17365D"
BLUE = "2F75B5"
PALE_BLUE = "D9EAF7"
PALE_GRAY = "F2F2F2"
PALE_RED = "FCE4D6"
WHITE = "FFFFFF"
TEXT = "1F1F1F"
GRAY = "666666"
GREEN = "548235"
RED = "C00000"
THIN_GRAY = Side(style="thin", color="D9E1F2")
MONEY = '#,##0.00;[Red](#,##0.00);-'
PERCENT = '0.00%'


def set_title(ws, title: str, subtitle: str, end_col: int = 12) -> None:
    ws.merge_cells(start_row=2, start_column=2, end_row=2, end_column=end_col)
    cell = ws.cell(2, 2, title)
    cell.font = Font(name="Arial", size=15, bold=True, color=NAVY)
    ws.merge_cells(start_row=3, start_column=2, end_row=3, end_column=end_col)
    cell = ws.cell(3, 2, subtitle)
    cell.font = Font(name="Arial", size=9, italic=True, color=GRAY)
    for col in range(2, end_col + 1):
        ws.cell(4, col).fill = PatternFill("solid", fgColor=NAVY)
    ws.row_dimensions[4].height = 3


def section(ws, row: int, start_col: int, end_col: int, text: str) -> None:
    ws.merge_cells(start_row=row, start_column=start_col, end_row=row, end_column=end_col)
    cell = ws.cell(row, start_col, text)
    cell.fill = PatternFill("solid", fgColor=NAVY)
    cell.font = Font(name="Arial", size=10, bold=True, color=WHITE)
    cell.alignment = Alignment(vertical="center")
    for col in range(start_col, end_col + 1):
        ws.cell(row, col).fill = PatternFill("solid", fgColor=NAVY)


def table_header(ws, row: int, start_col: int, labels: list[str]) -> None:
    for offset, label in enumerate(labels):
        cell = ws.cell(row, start_col + offset, label)
        cell.fill = PatternFill("solid", fgColor=BLUE)
        cell.font = Font(name="Arial", size=9, bold=True, color=WHITE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = Border(left=THIN_GRAY, right=THIN_GRAY, top=THIN_GRAY, bottom=THIN_GRAY)


def total_style(ws, row: int, start_col: int, end_col: int) -> None:
    for col in range(start_col, end_col + 1):
        cell = ws.cell(row, col)
        cell.fill = PatternFill("solid", fgColor=PALE_BLUE)
        cell.font = Font(name="Arial", size=10, bold=True, color=NAVY)
        cell.border = Border(top=Side(style="double", color=NAVY))


def apply_base(ws, max_row: int, max_col: int) -> None:
    ws.sheet_view.showGridLines = False
    for row in ws.iter_rows(min_row=1, max_row=max_row, min_col=1, max_col=max_col):
        for cell in row:
            if cell.font.name in (None, "Calibri"):
                cell.font = Font(name="Arial", size=10, color=TEXT)
            cell.alignment = Alignment(vertical="center", wrap_text=cell.alignment.wrap_text)


def build_report(result: dict[str, Any], output_path: str | Path) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    wb = Workbook()
    wb.remove(wb.active)
    ws_summary = wb.create_sheet("Resumen")
    ws_book = wb.create_sheet("Libro ventas SIAT")
    ws_back = wb.create_sheet("Respaldo IT SAP")
    ws_center = wb.create_sheet("Detalle CeBe")

    period = result["period"]
    totals = result["siat_totals"]
    gross = totals.get("IMPORTE TOTAL DE LA VENTA", 0.0)
    zero_rate = totals.get("VENTAS GRAVADAS A TASA CERO", 0.0)
    discounts = totals.get("DESCUENTOS BONIFICACIONES Y REBAJAS SUJETAS AL IVA", 0.0)
    vat_base = totals.get("IMPORTE BASE PARA DEBITO FISCAL", 0.0)
    vat_detail_sum = totals.get("DEBITO FISCAL", 0.0)
    # La determinación mensual aplica 13% sobre la base consolidada. La suma
    # del débito redondeado factura por factura se conserva solo como control.
    vat_siat = round(vat_base * 0.13, 2)

    composition = {row["group"]: row for row in result["composition"]}
    def comp_base(group: str) -> float:
        return composition.get(group, {}).get("base", 0.0)
    def comp_it(group: str) -> float:
        return composition.get(group, {}).get("it", 0.0)

    # Resumen vertical aprobado.
    set_title(ws_summary, "Determinación mensual de IVA e IT", f"{period['label']} | Fuente: SIAT y SAP", 12)
    section(ws_summary, 5, 2, 5, "DETERMINACIÓN DEL IVA")
    table_header(ws_summary, 6, 2, ["Concepto", "Cuenta / referencia", "Importe (Bs)", "Cálculo"])
    rows = [
        (8, "Ingresos facturados Universidad", "Ventas válidas menos Clínica y Souvenirs", gross - comp_base("Clínica odontológica") - comp_base("Souvenirs"), "Suma"),
        (9, "Ingresos Clínica odontológica", "420101006", comp_base("Clínica odontológica"), "Suma"),
        (10, "Ingresos por venta de souvenirs", "420101009", comp_base("Souvenirs"), "Suma"),
    ]
    ws_summary.merge_cells("B7:E7")
    ws_summary["B7"] = "más"
    ws_summary["B7"].fill = PatternFill("solid", fgColor=PALE_GRAY)
    ws_summary["B7"].font = Font(name="Arial", size=9, bold=True, color=NAVY)
    for row, concept, reference, amount, calculation in rows:
        ws_summary.cell(row, 2, concept)
        ws_summary.cell(row, 3, reference)
        ws_summary.cell(row, 4, amount)
        ws_summary.cell(row, 5, calculation)
    ws_summary.cell(11, 2, "TOTAL VENTAS VÁLIDAS SIAT")
    ws_summary.cell(11, 4, "=SUM(D8:D10)")
    ws_summary.cell(11, 5, "Suma")
    total_style(ws_summary, 11, 2, 5)
    ws_summary.merge_cells("B12:E12")
    ws_summary["B12"] = "menos"
    ws_summary["B12"].fill = PatternFill("solid", fgColor=PALE_GRAY)
    ws_summary["B12"].font = Font(name="Arial", size=9, bold=True, color=NAVY)
    ws_summary.cell(13, 2, "Descuentos, bonificaciones y rebajas")
    ws_summary.cell(13, 3, "Libro de ventas SIAT")
    ws_summary.cell(13, 4, discounts)
    ws_summary.cell(14, 2, "Ingresos por venta de libros")
    ws_summary.cell(14, 3, "Ventas gravadas a tasa cero")
    ws_summary.cell(14, 4, zero_rate)
    ws_summary.cell(15, 2, "TOTAL DEDUCCIONES")
    ws_summary.cell(15, 4, "=SUM(D13:D14)")
    ws_summary.cell(15, 5, "Resta")
    total_style(ws_summary, 15, 2, 5)
    ws_summary.cell(17, 2, "BASE IMPONIBLE PARA EL IVA")
    ws_summary.cell(17, 4, vat_base)
    ws_summary.cell(17, 5, "Según Libro de Ventas SIAT")
    total_style(ws_summary, 17, 2, 5)
    ws_summary.cell(18, 2, "DÉBITO FISCAL IVA")
    ws_summary.cell(18, 3, "13%")
    ws_summary.cell(18, 4, vat_siat)
    ws_summary.cell(18, 5, "Base × 13%")
    total_style(ws_summary, 18, 2, 5)

    section(ws_summary, 15, 8, 12, "CONCILIACIÓN IVA")
    table_header(ws_summary, 16, 8, ["Concepto", "SIAT", "Mayor SAP", "Diferencia"])
    ws_summary.cell(17, 8, "Débito fiscal neto")
    ws_summary.cell(17, 9, vat_siat)
    ws_summary.cell(17, 10, result["sap_vat_net"])
    ws_summary.cell(17, 11, "=J17-I17")
    total_style(ws_summary, 17, 8, 11)

    section(ws_summary, 21, 2, 6, "COMPOSICIÓN Y DETERMINACIÓN DEL IT SEGÚN SAP")
    table_header(ws_summary, 22, 2, ["Concepto", "Cuenta / referencia", "Base equivalente (Bs)", "IT SAP (Bs)", "% del IT"])
    refs = {
        "Clínica odontológica": "420101006",
        "Souvenirs": "420101009",
        "Otros ingresos": "420101001 + 420101010",
        "IAILC": "410207002",
        "IAFVV": "410207001",
    }
    labels = {
        "Clínica odontológica": "Ingresos Clínica odontológica",
        "Souvenirs": "Ingresos por venta de souvenirs",
        "Otros ingresos": "Otros ingresos",
        "IAILC": "Ingresos académicos - IAILC",
        "IAFVV": "Ingresos académicos - IAFVV",
    }
    for row, group in enumerate(COMPOSITION_GROUPS, start=24):
        ws_summary.cell(row, 2, labels[group])
        ws_summary.cell(row, 3, refs[group])
        ws_summary.cell(row, 4, comp_base(group))
        ws_summary.cell(row, 5, comp_it(group))
        ws_summary.cell(row, 6, f"=E{row}/$E$31")
    ws_summary.cell(30, 2, "BASE IMPONIBLE PARA EL IT")
    ws_summary.cell(30, 4, "=SUM(D24:D28)")
    total_style(ws_summary, 30, 2, 6)
    ws_summary.cell(31, 2, "IMPUESTO A LAS TRANSACCIONES")
    ws_summary.cell(31, 3, "3%")
    ws_summary.cell(31, 4, "=D30")
    ws_summary.cell(31, 5, "=SUM(E24:E28)")
    ws_summary.cell(31, 6, "=SUM(F24:F28)")
    total_style(ws_summary, 31, 2, 6)

    ws_summary.merge_cells("B37:L39")
    ws_summary["B37"] = (
        "El SIAT sustenta las ventas válidas, descuentos y libros a tasa cero. "
        "SAP sustenta el IVA contabilizado y la composición del IT. Los reversos y anulaciones "
        "se conservan con su signo y se evalúan por su efecto neto."
    )
    ws_summary["B37"].fill = PatternFill("solid", fgColor=PALE_GRAY)
    ws_summary["B37"].font = Font(name="Arial", size=9, italic=True, color=GRAY)
    ws_summary["B37"].alignment = Alignment(wrap_text=True, vertical="center")

    for row in range(1, 40):
        for col in (4, 5, 9, 10, 11):
            ws_summary.cell(row, col).number_format = MONEY
    for row in range(24, 32):
        ws_summary.cell(row, 6).number_format = PERCENT
    widths = {1: 3, 2: 38, 3: 29, 4: 19, 5: 17, 6: 13, 7: 3, 8: 16, 9: 16, 10: 16, 11: 16, 12: 5}
    for col, width in widths.items():
        ws_summary.column_dimensions[get_column_letter(col)].width = width
    ws_summary.sheet_properties.tabColor = NAVY
    ws_summary.page_setup.orientation = "landscape"
    ws_summary.page_setup.fitToWidth = 1
    ws_summary.sheet_properties.pageSetUpPr.fitToPage = True
    ws_summary.print_area = "B2:L39"

    # Libro SIAT con el IT inmediatamente después del débito fiscal.
    source_columns = [column for column in result["siat"].columns if not column.startswith("_")]
    debit_index = source_columns.index("DEBITO FISCAL") + 1
    output_columns = source_columns[:debit_index] + ["IT SAP"] + source_columns[debit_index:]
    set_title(ws_book, "Libro de Ventas SIAT", f"{period['label']} | Detalle original y asignación del IT SAP", len(output_columns))
    summary_labels = [
        (5, "Total ventas válidas", gross),
        (6, "Venta de libros - tasa cero", zero_rate),
        (7, "Descuentos", discounts),
        (8, "Base para débito fiscal", vat_base),
        (9, "Débito fiscal sobre base consolidada", vat_siat),
        (10, "IT SAP asignado", result["it_assigned_total"]),
    ]
    for row, label, value in summary_labels:
        ws_book.cell(row, 2, label)
        ws_book.cell(row, 3, value)
        ws_book.cell(row, 3).number_format = MONEY
    total_style(ws_book, 10, 2, 3)
    counts = result["siat_counts"]
    control_labels = [
        (5, "Registros del libro", counts["all"], False),
        (6, "Comprobantes válidos", counts["valid"], False),
        (7, "Comprobantes anulados", counts["annulled"], False),
        (8, "Débito fiscal por comprobante", vat_detail_sum, True),
        (9, "Diferencia de asignación IT", result["assignment_difference"], True),
        (10, "Grupos de reversión neto cero", result["zero_groups"], False),
    ]
    for row, label, value, is_money in control_labels:
        ws_book.cell(row, 5, label)
        ws_book.cell(row, 6, value)
        if is_money:
            ws_book.cell(row, 6).number_format = MONEY
    total_style(ws_book, 10, 5, 6)
    header_row = 13
    table_header(ws_book, header_row, 1, output_columns)
    source = result["siat"]
    it_series = result["it_per_row"]
    for out_row, (index, record) in enumerate(source.iterrows(), start=header_row + 1):
        values = []
        for column in output_columns:
            if column == "IT SAP":
                value = float(it_series.at[index])
            else:
                value = record[column]
                if pd.isna(value):
                    value = None
                elif isinstance(value, pd.Timestamp):
                    value = value.to_pydatetime()
                elif hasattr(value, "item"):
                    try:
                        value = value.item()
                    except Exception:
                        pass
            values.append(value)
        for col, value in enumerate(values, start=1):
            ws_book.cell(out_row, col, value)
    ws_book.auto_filter.ref = f"A{header_row}:{get_column_letter(len(output_columns))}{header_row + len(source)}"
    ws_book.freeze_panes = f"A{header_row + 1}"
    money_names = {
        "IMPORTE TOTAL DE LA VENTA", "IMPORTE ICE", "IMPORTE IEHD", "IMPORTE IPJ", "TASAS",
        "OTROS NO SUJETOS AL IVA", "EXPORTACIONES Y OPERACIONES EXENTAS", "VENTAS GRAVADAS A TASA CERO",
        "SUBTOTAL", "DESCUENTOS BONIFICACIONES Y REBAJAS SUJETAS AL IVA", "IMPORTE GIFT CARD",
        "IMPORTE BASE PARA DEBITO FISCAL", "DEBITO FISCAL", "IT SAP",
    }
    for col, name in enumerate(output_columns, start=1):
        width = 14
        if name in {"NOMBRE O RAZON SOCIAL", "CODIGO DE AUTORIZACIÓN"}:
            width = 31
        elif name in {"SEDE SEGÚN NETVALLE", "CAJA / PUNTO SEGÚN NETVALLE"}:
            width = 21
        elif len(name) > 26:
            width = 20
        ws_book.column_dimensions[get_column_letter(col)].width = width
        if name in money_names:
            for row in range(header_row + 1, header_row + len(source) + 1):
                ws_book.cell(row, col).number_format = MONEY
    ws_book.sheet_properties.tabColor = "70AD47"

    # Respaldo del IT por naturaleza y sede.
    set_title(ws_back, "Composición del IT según SAP", f"{period['label']} | Asientos, cuentas de ingreso y centros de beneficio", 11)
    section(ws_back, 5, 2, 11, "COMPOSICIÓN POR NATURALEZA Y SEDE")
    table_header(ws_back, 6, 2, ["Grupo", *SITE_ORDER, "IT SAP", "Base equivalente", "% del IT"])
    for row, item in enumerate(result["composition"], start=7):
        ws_back.cell(row, 2, item["group"])
        for offset, site in enumerate(SITE_ORDER, start=3):
            ws_back.cell(row, offset, item["by_site"].get(site, 0.0))
        ws_back.cell(row, 8, item["it"])
        ws_back.cell(row, 9, item["base"])
        ws_back.cell(row, 10, item["it"] / result["it_source_total"] if result["it_source_total"] else 0)
    total_row = 7 + len(result["composition"])
    ws_back.cell(total_row, 2, "TOTAL")
    for col in range(3, 10):
        ws_back.cell(total_row, col, f"=SUM({get_column_letter(col)}7:{get_column_letter(col)}{total_row - 1})")
    ws_back.cell(total_row, 10, f"=SUM(J7:J{total_row - 1})")
    total_style(ws_back, total_row, 2, 10)
    for row in range(7, total_row + 1):
        for col in range(3, 10):
            ws_back.cell(row, col).number_format = MONEY
        ws_back.cell(row, 10).number_format = PERCENT

    control_start = total_row + 3
    section(ws_back, control_start, 2, 7, "CONTROL DEL IT Y REVERSOS")
    table_header(ws_back, control_start + 1, 2, ["IT mayor SAP", "IT asignado al SIAT", "Diferencia", "Grupos neto cero", "Glosas sin factura", "Excepciones"])
    controls = [
        result["it_source_total"], result["it_assigned_total"], result["assignment_difference"],
        result["zero_groups"],
        sum(1 for item in result["exceptions"] if item["tipo"].startswith("Glosa")),
        len(result["exceptions"]),
    ]
    for col, value in enumerate(controls, start=2):
        ws_back.cell(control_start + 2, col, value)
        if col <= 4:
            ws_back.cell(control_start + 2, col).number_format = MONEY
    for col, width in {1: 3, 2: 30, 3: 18, 4: 18, 5: 18, 6: 18, 7: 18, 8: 18, 9: 20, 10: 16, 11: 4}.items():
        ws_back.column_dimensions[get_column_letter(col)].width = width

    # Una sola visual: cada barra es una sede y cada color una naturaleza del IT.
    chart = BarChart()
    chart.type = "bar"
    chart.style = 10
    chart.grouping = "stacked"
    chart.overlap = 100
    chart.title = "IT por sede y concepto (Bs)"
    chart.x_axis.title = "IT (Bs)"
    chart.legend.position = "b"
    chart.height = 7.0
    chart.width = 13.0
    chart.add_data(
        Reference(ws_back, min_col=2, max_col=7, min_row=7, max_row=7 + len(result["composition"]) - 1),
        titles_from_data=True,
        from_rows=True,
    )
    chart.set_categories(Reference(ws_back, min_col=3, max_col=7, min_row=6, max_row=6))
    series_colors = ["1F4E78", "70AD47", "5B9BD5", "ED7D31", "A5A5A5"]
    for series, color in zip(chart.series, series_colors):
        series.graphicalProperties.solidFill = color
        series.graphicalProperties.line.solidFill = color
    ws_summary.add_chart(chart, "H21")

    # Detalle CeBe agrupado: subtotal visible por sede y centros desplegables.
    set_title(ws_center, "Detalle del IT por sede y centro de beneficio", f"{period['label']} | Débitos, créditos, reversiones y neto", 10)
    section(ws_center, 5, 1, 10, "TOTALIZADO POR SEDE | EXPANDA EL GRUPO PARA VER SUS CENTROS")
    center_headers = ["Sede / CeBe", "Denominación", "Área", "Movimientos", "Débitos", "Créditos / reversos", "IT neto", "Base equivalente", "% del IT", "Bloqueado"]
    table_header(ws_center, 6, 1, center_headers)
    center_by_site = defaultdict(list)
    for item in result["center_rows"]:
        center_by_site[item["site"]].append(item)
    all_sites = SITE_ORDER + sorted(site for site in center_by_site if site not in SITE_ORDER)
    row = 7
    for site in all_sites:
        items = center_by_site.get(site, [])
        site_debits = sum(item["debits"] for item in items)
        site_credits = sum(item["credits"] for item in items)
        site_net = sum(item["net"] for item in items)
        subtotal_row = row
        subtotal_values = [
            site, f"{len(items)} centros", "", sum(item["moves"] for item in items),
            site_debits, site_credits, site_net, site_net / 0.03 if site_net else 0.0,
            site_net / result["it_source_total"] if result["it_source_total"] else 0.0,
            sum(1 for item in items if str(item["blocked"]).strip().upper() not in {"", "NO", "FALSE", "0"}),
        ]
        for col, value in enumerate(subtotal_values, start=1):
            ws_center.cell(subtotal_row, col, value)
        total_style(ws_center, subtotal_row, 1, 10)
        row += 1
        detail_start = row
        for item in sorted(items, key=lambda value: (-abs(value["net"]), str(value["center"]))):
            values = [
                item["center"], item["name"], item["area"], item["moves"], item["debits"],
                item["credits"], item["net"], item["base"],
                item["net"] / result["it_source_total"] if result["it_source_total"] else 0.0,
                item["blocked"],
            ]
            for col, value in enumerate(values, start=1):
                ws_center.cell(row, col, value)
            row += 1
        if items:
            ws_center.row_dimensions.group(detail_start, row - 1, outline_level=1, hidden=True)
    grand_total_row = row
    grand_values = [
        "TOTAL GENERAL", f"{len(result['center_rows'])} centros", "", sum(item["moves"] for item in result["center_rows"]),
        sum(item["debits"] for item in result["center_rows"]),
        sum(item["credits"] for item in result["center_rows"]),
        sum(item["net"] for item in result["center_rows"]),
        sum(item["base"] for item in result["center_rows"]), 1.0,
        sum(1 for item in result["center_rows"] if str(item["blocked"]).strip().upper() not in {"", "NO", "FALSE", "0"}),
    ]
    for col, value in enumerate(grand_values, start=1):
        ws_center.cell(grand_total_row, col, value)
    total_style(ws_center, grand_total_row, 1, 10)
    for data_row in range(7, grand_total_row + 1):
        for col in range(5, 9):
            ws_center.cell(data_row, col).number_format = MONEY
        ws_center.cell(data_row, 9).number_format = PERCENT
    ws_center.sheet_properties.outlinePr.summaryBelow = False
    ws_center.freeze_panes = "A7"
    widths = [22, 40, 20, 14, 16, 20, 16, 18, 13, 12]
    for col, width in enumerate(widths, start=1):
        ws_center.column_dimensions[get_column_letter(col)].width = width
    ws_center.page_setup.orientation = "landscape"
    ws_center.print_area = f"A2:J{grand_total_row}"

    # Solo aparece si existe algo que requiere revisión humana.
    if result["exceptions"] or result["unknown_income_accounts"] or abs(result["assignment_difference"]) >= 0.005:
        ws_exc = wb.create_sheet("Excepciones")
        set_title(ws_exc, "Excepciones para revisión", f"{period['label']} | El programa no fuerza coincidencias dudosas", 10)
        headers = ["Tipo", "Factura", "Fecha", "IT SAP", "Candidatos válidos", "Candidatos anulados", "Cajas SAP", "Segmentos SAP", "Glosa"]
        table_header(ws_exc, 6, 1, headers)
        all_exceptions = list(result["exceptions"])
        for account in result["unknown_income_accounts"]:
            all_exceptions.append({
                "tipo": "Cuenta de ingreso sin regla tributaria",
                "factura": account,
                "fecha": "",
                "it_sap": 0.0,
                "candidatos_validos": "",
                "candidatos_anulados": "",
                "cajas_sap": "",
                "segmentos_sap": "",
                "glosa": "Agregar la cuenta al maestro antes de cerrar el reporte.",
            })
        for row_number, item in enumerate(all_exceptions, start=7):
            values = [
                item.get("tipo"), item.get("factura"), item.get("fecha"), item.get("it_sap"),
                item.get("candidatos_validos"), item.get("candidatos_anulados"), item.get("cajas_sap"),
                item.get("segmentos_sap"), item.get("glosa"),
            ]
            for col, value in enumerate(values, start=1):
                ws_exc.cell(row_number, col, value)
            ws_exc.cell(row_number, 4).number_format = MONEY
        ws_exc.auto_filter.ref = f"A6:I{6 + len(all_exceptions)}"
        ws_exc.freeze_panes = "A7"
        for col, width in enumerate([42, 16, 14, 16, 18, 18, 25, 18, 60], start=1):
            ws_exc.column_dimensions[get_column_letter(col)].width = width
        ws_exc.sheet_properties.tabColor = RED

    for ws in wb.worksheets:
        max_row = max(ws.max_row, 1)
        max_col = max(ws.max_column, 1)
        apply_base(ws, max_row, max_col)
        ws.sheet_properties.pageSetUpPr.fitToPage = True
        ws.page_setup.fitToWidth = 1
        ws.page_margins.left = 0.25
        ws.page_margins.right = 0.25
        ws.page_margins.top = 0.5
        ws.page_margins.bottom = 0.5

    # Fórmulas se recalculan al abrir en Excel.
    try:
        wb.calculation.fullCalcOnLoad = True
        wb.calculation.forceFullCalc = True
        wb.calculation.calcMode = "auto"
    except Exception:
        pass
    wb.save(output_path)
    return output_path


def run_automation(
    siat_path: str | Path,
    sap_path: str | Path,
    output_path: str | Path,
    cebe_map: dict[str, dict[str, Any]] | None = None,
) -> tuple[Path, dict[str, Any]]:
    siat, sap = load_inputs(siat_path, sap_path)
    result = process_data(siat, sap, cebe_map=cebe_map)
    output = build_report(result, output_path)
    return output, result

# Maestros permanentes incorporados. No se cargan cada mes.
import json
CEBE_MAP = json.loads("{\"100101\":{\"denominacion\":\"DISTRIBUIBLES - CBB\",\"descripcion\":\"DISTRIBUIBLES - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"\",\"bloqueado\":false},\"10010101\":{\"denominacion\":\"ADMINISTRACIÓN - CBB\",\"descripcion\":\"ADM. - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010102\":{\"denominacion\":\"MANTENIMIENTO - CBB\",\"descripcion\":\"MNT. - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010103\":{\"denominacion\":\"INVESTIGACIÓN - CBB\",\"descripcion\":\"INV. - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010104\":{\"denominacion\":\"EXTENSIÓN - CBB\",\"descripcion\":\"EXT. - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010105\":{\"denominacion\":\"MARKETING - CBB\",\"descripcion\":\"MKT. - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010106\":{\"denominacion\":\"CAF - CBB\",\"descripcion\":\"CAF - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010107\":{\"denominacion\":\"GASTOS ACADÉMICOS - CBB\",\"descripcion\":\"G.ACAD - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010108\":{\"denominacion\":\"DIRECCIÓN NACIONAL - CBB\",\"descripcion\":\"DIR.NAL - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010109\":{\"denominacion\":\"SUELDOS ACADÉMICOS - CBB\",\"descripcion\":\"SUEL. ACAD - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010110\":{\"denominacion\":\"LABORATORIO GENERAL - CBB\",\"descripcion\":\"LAB. GRAL - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010111\":{\"denominacion\":\"SUELDOS ADMINISTRATIVOS POSTGRADO - CBB\",\"descripcion\":\"S. ADM. POSTG - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010112\":{\"denominacion\":\"NO APLICABLES A CARRERAS - CBB\",\"descripcion\":\"NO APLICA CAR - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010113\":{\"denominacion\":\"DISTRIBUIBLE POSTGRADO - CBB\",\"descripcion\":\"DIST. POSTGR - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010114\":{\"denominacion\":\"ACTIVOS ACADÉMICOS DE USO GENERAL - CBB\",\"descripcion\":\"ACT ACAD GRAL - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010115\":{\"denominacion\":\"DEPÓSITO TRANSITORIO - CBB\",\"descripcion\":\"DEP. TRANSITO - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010116\":{\"denominacion\":\"DIRECTORES Y COORDINADORES CARR - CBB\",\"descripcion\":\"DIREC Y COORD - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010117\":{\"denominacion\":\"SOUVENIRS BOUTIQUE - CBB\",\"descripcion\":\"SOUVENIR BOUTQ - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010118\":{\"denominacion\":\"PROYECTO HELVETAS - CBB\",\"descripcion\":\"PRO. HELVETAS - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10010119\":{\"denominacion\":\"PROYECTO CHORRILLOS - CBB\",\"descripcion\":\"PRO CHORRILLOS - CBB\",\"departamento\":\"COCHABAMBA\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"100102\":{\"denominacion\":\"CIENCIAS DE LA SALUD - CBB\",\"descripcion\":\"CS.SALUD - CBB\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"\",\"bloqueado\":false},\"10010201\":{\"denominacion\":\"MEDICINA - CBB\",\"descripcion\":\"MED - CBB\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010202\":{\"denominacion\":\"ODONTOLOGÍA - CBB\",\"descripcion\":\"ODO - CBB\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010203\":{\"denominacion\":\"BIOQUÍMICA Y FARMACIA - CBB\",\"descripcion\":\"BYF - CBB\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010204\":{\"denominacion\":\"FISIOTERAPIA Y KINESIOLOGÍA - CBB\",\"descripcion\":\"LFK - CBB\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010205\":{\"denominacion\":\"ENFERMERÍA CLINICO QUIRURGICA - CBB\",\"descripcion\":\"ECQ - CBB\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010206\":{\"denominacion\":\"NUTRICIÓN Y DIETÉTICA - CBB\",\"descripcion\":\"LND - CBB\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100103\":{\"denominacion\":\"CS. EMPRESARIALES Y SOCIALES - CBB\",\"descripcion\":\"CS.EMP Y SOC - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"\",\"bloqueado\":false},\"10010301\":{\"denominacion\":\"ING. COMERCIAL - CBB\",\"descripcion\":\"ICO - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010302\":{\"denominacion\":\"ING. COMERCIO INTERNACIONAL - CBB\",\"descripcion\":\"ICT - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010303\":{\"denominacion\":\"DERECHO Y CIENCIAS JURÍDICAS - CBB\",\"descripcion\":\"LDE - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010304\":{\"denominacion\":\"COMUNICACIÓN Y M. DIGITALES - CBB\",\"descripcion\":\"LCM - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010305\":{\"denominacion\":\"ADMINISTRACIÓN DE EMPRESAS - CBB\",\"descripcion\":\"LAE - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010306\":{\"denominacion\":\"PSICOLOGÍA - CBB\",\"descripcion\":\"LPS - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010307\":{\"denominacion\":\"CONTADURÍA PÚBLICA - CBB\",\"descripcion\":\"LCN - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010308\":{\"denominacion\":\"ING. FINANCIERA Y RIESGOS - CBB\",\"descripcion\":\"IFR - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010309\":{\"denominacion\":\"ING. CIENCIA DATOS E INT. NEGOCIOS - CBB\",\"descripcion\":\"LCD - CBB\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100104\":{\"denominacion\":\"INFORMÁTICA Y ELECTRÓNICA - CBB\",\"descripcion\":\"INF Y ELEC - CBB\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"\",\"bloqueado\":false},\"10010401\":{\"denominacion\":\"ING. BIOMÉDICA - CBB\",\"descripcion\":\"IBI - CBB\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010402\":{\"denominacion\":\"ING. ELECTRÓNICA - CBB\",\"descripcion\":\"IEL - CBB\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010403\":{\"denominacion\":\"ING. ELECTRO Y DE SISTEMAS - CBB\",\"descripcion\":\"IES - CBB\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010404\":{\"denominacion\":\"ING. TELECOMUNICACIONES - CBB\",\"descripcion\":\"IET - CBB\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010405\":{\"denominacion\":\"ING. SISTEMAS INFORMÁTICOS - CBB\",\"descripcion\":\"ISI - CBB\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010406\":{\"denominacion\":\"TEC. SUP. VIDEO JUEGOS - CBB\",\"descripcion\":\"TSV - CBB\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100105\":{\"denominacion\":\"ARQUITECTURA Y TURISMO - CBB\",\"descripcion\":\"ARQ Y TURIS - CBB\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"\",\"bloqueado\":false},\"10010501\":{\"denominacion\":\"ARQUITECTURA Y URBANISMO - CBB\",\"descripcion\":\"ARQ - CBB\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010502\":{\"denominacion\":\"TURISMO Y HOTELERÍA - CBB\",\"descripcion\":\"LTH - CBB\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010503\":{\"denominacion\":\"GASTRONOMÍA - CBB\",\"descripcion\":\"LGS - CBB\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010504\":{\"denominacion\":\"DIS. INTERIORES Y PAISAJIS - CBB\",\"descripcion\":\"DYP - CBB\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010505\":{\"denominacion\":\"DISEÑO GRÁFICO Y COMUNIC. VISUAL - CBB\",\"descripcion\":\"LDG - CBB\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100106\":{\"denominacion\":\"TECNOLOGÍA - CBB\",\"descripcion\":\"TECNO - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"\",\"bloqueado\":false},\"10010601\":{\"denominacion\":\"ING. PETROQUÍMICA - CBB\",\"descripcion\":\"IPQ - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010602\":{\"denominacion\":\"ING. CIVIL - CBB\",\"descripcion\":\"ICI - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010603\":{\"denominacion\":\"ING. PETRÓLEO, GAS Y ENERG. - CBB\",\"descripcion\":\"IPG - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010604\":{\"denominacion\":\"ING. INDUSTRIAS ALIMENTARIAS - CBB\",\"descripcion\":\"IIA - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010605\":{\"denominacion\":\"ING. AERONÁUTICA - CBB\",\"descripcion\":\"IAE - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010606\":{\"denominacion\":\"ING. ELECTROMECÁNICA - CBB\",\"descripcion\":\"IEL - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010607\":{\"denominacion\":\"ING. MECÁNICA Y AUTOM. INDUSTRIAL - CBB\",\"descripcion\":\"IMT - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010608\":{\"denominacion\":\"ING. INDUSTRIAL Y SISTEMAS - CBB\",\"descripcion\":\"IIS - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010609\":{\"denominacion\":\"ING. INDUSTRIAL - CBB\",\"descripcion\":\"IIN - CBB\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100107\":{\"denominacion\":\"POSTGRADO - CBB\",\"descripcion\":\"POSTGRADO - CBB\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"\",\"bloqueado\":false},\"10010701\":{\"denominacion\":\"DOCTORADO - CBB\",\"descripcion\":\"DOCTORADO - CBB\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010702\":{\"denominacion\":\"MAESTRÍA - CBB\",\"descripcion\":\"MAESTRIA - CBB\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010703\":{\"denominacion\":\"DIPLOMADO - CBB\",\"descripcion\":\"DIPLOMADO - CBB\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10010704\":{\"denominacion\":\"CURSOS - CBB\",\"descripcion\":\"CURSOS - CBB\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100108\":{\"denominacion\":\"LABORATORIOS - CBB\",\"descripcion\":\"LABORATORIOS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"\",\"bloqueado\":false},\"10010801\":{\"denominacion\":\"N/A DE RADIO - CBB\",\"descripcion\":\"SET RADIO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10010802\":{\"denominacion\":\"SET DE FOTOGRAFIA - CBB\",\"descripcion\":\"FOTOGRAFIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010803\":{\"denominacion\":\"MUSEO GENERAL - CBB\",\"descripcion\":\"MUSEO GRAL - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010804\":{\"denominacion\":\"LAB. DE RECURSOS FISICOS - CBB\",\"descripcion\":\"REC FISI - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010805\":{\"denominacion\":\"LAB. DE ENTRENAMIENTO - CBB\",\"descripcion\":\"ENTRENAM - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010806\":{\"denominacion\":\"KINESIOLOGIA - CBB\",\"descripcion\":\"KINESIO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010807\":{\"denominacion\":\"LAB. DE SIMULACIÓN - CBB\",\"descripcion\":\"SIMULACION - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010808\":{\"denominacion\":\"LAB. DE NEUROLOGÍA - CBB\",\"descripcion\":\"NEURO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010809\":{\"denominacion\":\"LAB. DE PSICOMOTRICIDAD - CBB\",\"descripcion\":\"PSICOMO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010810\":{\"denominacion\":\"LAB. HISTOLOGÍA Y PATOLOGÍA - CBB\",\"descripcion\":\"HIST Y PATO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010811\":{\"denominacion\":\"LAB. GENETICA Y EMBRIOLOGÍA - CBB\",\"descripcion\":\"GEN Y EMBRIO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010812\":{\"denominacion\":\"LAB. QUÍMICA Y BIOQUÍMICA - CBB\",\"descripcion\":\"QUI Y BIOQUI - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010813\":{\"denominacion\":\"LAB. DE HEMATOLOGÍA - CBB\",\"descripcion\":\"HEMATO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010814\":{\"denominacion\":\"SALA DE BALANZAS - CBB\",\"descripcion\":\"S.BALANZAS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010815\":{\"denominacion\":\"SALA DE EQUIPOS - CBB\",\"descripcion\":\"S.EQUIPOS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010816\":{\"denominacion\":\"LAB. MICROBIO Y PARASITO - CBB\",\"descripcion\":\"MICRO Y PARAS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010817\":{\"denominacion\":\"LAB. SEMIOLOGÍA Y FISIOLOGÍA - CBB\",\"descripcion\":\"SEMI Y FISIO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010818\":{\"denominacion\":\"CIRUGIA - CBB\",\"descripcion\":\"CIRUGIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010819\":{\"denominacion\":\"ODONTOPEDRIATRÍA - CBB\",\"descripcion\":\"ODONTOPEDIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010820\":{\"denominacion\":\"LAB. DE PROTESIS - CBB\",\"descripcion\":\"PROTESIS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010821\":{\"denominacion\":\"QUIROFANO - CBB\",\"descripcion\":\"QUIROFANO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010822\":{\"denominacion\":\"SALA DE ESTERILIZACIÓN - CBB\",\"descripcion\":\"S.ESTERILI - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010823\":{\"denominacion\":\"CLINICA (A) - CBB\",\"descripcion\":\"CLINICA A - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010824\":{\"denominacion\":\"CLINICA (B) - CBB\",\"descripcion\":\"CLINICA B - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10010825\":{\"denominacion\":\"SALA DE ADMICIÓN (CIRUGIA) - CBB\",\"descripcion\":\"CIR. ADMISION - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010826\":{\"denominacion\":\"SALA RADIOLOGÍA (PERIEPI. PANORA) - CBB\",\"descripcion\":\"RADIOLOGIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010827\":{\"denominacion\":\"MORFOLOGÍA SALA DE ANATOMIA - CBB\",\"descripcion\":\"MORFOLOGIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010828\":{\"denominacion\":\"OSTEOTECA - CBB\",\"descripcion\":\"OSTEOTECA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010829\":{\"denominacion\":\"SALA DE INVESTIGACIÓN - CBB\",\"descripcion\":\"S. INVESTI - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010830\":{\"denominacion\":\"MUSEO DE ANATOMÍA - CBB\",\"descripcion\":\"MUSEO ATO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010831\":{\"denominacion\":\"SALA ANATOMÍA Y SIMULACIÓN - CBB\",\"descripcion\":\"ANATO Y SIM - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010832\":{\"denominacion\":\"DEPOSITO PIEZAS ANATOMICAS - CBB\",\"descripcion\":\"DEP PIEZ ANAT - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010833\":{\"denominacion\":\"LAB. DE PASTELERIA - CBB\",\"descripcion\":\"PASTELERIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010834\":{\"denominacion\":\"LAB. DE COCINA - CBB\",\"descripcion\":\"COCINA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010835\":{\"denominacion\":\"ECONOMATO - CBB\",\"descripcion\":\"ECONOMATO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010836\":{\"denominacion\":\"LAB. DE PANADERIA - CBB\",\"descripcion\":\"PANADERIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010837\":{\"denominacion\":\"COCTELERIA - CBB\",\"descripcion\":\"COCTELERIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010838\":{\"denominacion\":\"SALA DE PEDIATRÍA - CBB\",\"descripcion\":\"S. PEDIATRIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010839\":{\"denominacion\":\"SALA GINECO. Y OBSTETRICIA - CBB\",\"descripcion\":\"S. GINE Y OBS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010840\":{\"denominacion\":\"SALA DE SUMINISTROS - CBB\",\"descripcion\":\"S. SUMINISTRO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010841\":{\"denominacion\":\"SALA DE QUIROFANO - CBB\",\"descripcion\":\"S. QUIROFANO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010842\":{\"denominacion\":\"SALA DE HOSPITALIZACIÓN - CBB\",\"descripcion\":\"S. HOSPITAL - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010843\":{\"denominacion\":\"SALA DE TERAPIA INTENSIVA - CBB\",\"descripcion\":\"S. TER INTENS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010844\":{\"denominacion\":\"SALA DE ATENCIÓN PRIMARIA - CBB\",\"descripcion\":\"S. ATEN PRIM - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010845\":{\"denominacion\":\"SALA MULTIPROPOSITO - CBB\",\"descripcion\":\"S. MULTIPROP - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010846\":{\"denominacion\":\"TALLER DE ARQUITECTURA - CBB\",\"descripcion\":\"T. ARQUITEC - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010847\":{\"denominacion\":\"GB. URBANISMO Y TRAFICO - CBB\",\"descripcion\":\"G. URB Y TRAF - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010848\":{\"denominacion\":\"INCUBACIÓN - CBB\",\"descripcion\":\"INCUBACION - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010849\":{\"denominacion\":\"SALA DE JUICIOS ORALES - CBB\",\"descripcion\":\"S. JUICIOS OR - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010850\":{\"denominacion\":\"NEUROMARKETING - CBB\",\"descripcion\":\"NEUROMKT - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010851\":{\"denominacion\":\"CONSERVATORIO Y SALA REUNIONES - CBB\",\"descripcion\":\"CONSER Y REUN - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010852\":{\"denominacion\":\"INNOVACIÓN - CBB\",\"descripcion\":\"INNOVACION - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010853\":{\"denominacion\":\"SALA DE REUNIONES - CBB\",\"descripcion\":\"S. REUNION - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010854\":{\"denominacion\":\"CAMARA GESELL - CBB\",\"descripcion\":\"CAM GESELL - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010855\":{\"denominacion\":\"CENTRO DE COMPUTO - CBB\",\"descripcion\":\"CEN COMPUTO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010856\":{\"denominacion\":\"LAB. DE BURSATIL - CBB\",\"descripcion\":\"BURSATIL - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010857\":{\"denominacion\":\"LAB. DISEÑO COMPUTARIZADO - CBB\",\"descripcion\":\"DIS COMPUTA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010858\":{\"denominacion\":\"LAB. DE FISICA - CBB\",\"descripcion\":\"FISICA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010859\":{\"denominacion\":\"LAB. ELECTRÓNICA DIGITAL - CBB\",\"descripcion\":\"ELECT DIGITAL - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010860\":{\"denominacion\":\"LAB. DE ROBOTICA - CBB\",\"descripcion\":\"ROBOTICA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010861\":{\"denominacion\":\"LAB. TELECOMUNICACIONES - CBB\",\"descripcion\":\"L. TELECOM - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010862\":{\"denominacion\":\"LAB. ELECTRÓNICA Y BIOMÉDICA - CBB\",\"descripcion\":\"L. ELEC Y BIO - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010863\":{\"denominacion\":\"LAB. ENERGIAS ALTERNATIVAS - CBB\",\"descripcion\":\"ENER ALTER - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010864\":{\"denominacion\":\"LAB. LUBRICANTES Y CARBURANTES - CBB\",\"descripcion\":\"LUBR Y CAR - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010865\":{\"denominacion\":\"LAB. SIMULACIÓN INDUS. INGENI MET - CBB\",\"descripcion\":\"SIM INDUSTRIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010866\":{\"denominacion\":\"LAB. ANL. INSTRUMEN, GB METRO IND - CBB\",\"descripcion\":\"A IND ING MET - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010867\":{\"denominacion\":\"GB. SEGURIDAD INDUST. Y TOPOG. - CBB\",\"descripcion\":\"SEG IND Y TOP - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010868\":{\"denominacion\":\"LAB. DE SANITARIA - CBB\",\"descripcion\":\"SANITARIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010869\":{\"denominacion\":\"LAB. HIDRAULICA - CBB\",\"descripcion\":\"HIDRAULICA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010870\":{\"denominacion\":\"LAB. SUELOS, HORMIGONES Y ASFALTO - CBB\",\"descripcion\":\"SUELOS HORMIG - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010871\":{\"denominacion\":\"LAB. RESISTENCIA MATERIALES - CBB\",\"descripcion\":\"RESIS MATER - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010872\":{\"denominacion\":\"LAB. PROCESOS INDUSTRIALES - CBB\",\"descripcion\":\"PROC INDUS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010873\":{\"denominacion\":\"LAB. ANALISIS ESTRUCTURAL - CBB\",\"descripcion\":\"ANALIS ESTRUC - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010874\":{\"denominacion\":\"LAB. DE DESTILACIÓN - CBB\",\"descripcion\":\"DESTILACION - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010875\":{\"denominacion\":\"LAB. CONDUCTUAL - CBB\",\"descripcion\":\"CONDUCTUAL - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010876\":{\"denominacion\":\"LAB. PLANTA DE ALIMENTOS - CBB\",\"descripcion\":\"PLANTA ALIM. - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010877\":{\"denominacion\":\"LAB. EMBUTIDOS - CBB\",\"descripcion\":\"EMBUTIDOS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010878\":{\"denominacion\":\"LAB. AUDIO DIGITAL - CBB\",\"descripcion\":\"AUDIO DIGITAL - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010879\":{\"denominacion\":\"LAB. SET DE TELEVISION - CBB\",\"descripcion\":\"ST TELEVISION - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010880\":{\"denominacion\":\"LAB QUIMICA ORGANICA E INORGANICA - CBB\",\"descripcion\":\"QMC ORG/INORG - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010881\":{\"denominacion\":\"LAB. FISICOQUIMICA - CBB\",\"descripcion\":\"FISICOQUIMICA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010882\":{\"denominacion\":\"LAB. PARASITOLOGIA - CBB\",\"descripcion\":\"PARASITOLOGIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010883\":{\"denominacion\":\"LAB. ELECTROTENIA - CBB\",\"descripcion\":\"ELECTROTENIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010884\":{\"denominacion\":\"LAB. POTENCIA - CBB\",\"descripcion\":\"POTENCIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010885\":{\"denominacion\":\"LAB. MAQUINAS ELECTRICAS - CBB\",\"descripcion\":\"MAQ. ELECTR. - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010886\":{\"denominacion\":\"LAB. MAQUINAS TERMICAS - CBB\",\"descripcion\":\"MAQ. TERMIC. - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010887\":{\"denominacion\":\"LAB. PROCESOS - CBB\",\"descripcion\":\"PROCESOS - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010888\":{\"denominacion\":\"LAB. ANALISIS SENSORIAL - CBB\",\"descripcion\":\"ANALIS SENSOR - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010889\":{\"denominacion\":\"LAB. OBRAS HIDRAULICAS - CBB\",\"descripcion\":\"OBRAS HIDRAU - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010890\":{\"denominacion\":\"LAB. AUTOMOTORES - CBB\",\"descripcion\":\"AUTOMOTORES - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010891\":{\"denominacion\":\"TALLER DE METALMECANICA - CBB\",\"descripcion\":\"METALMECANICA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010892\":{\"denominacion\":\"BIBLIOTECA - CBB\",\"descripcion\":\"BIBLIOTECA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010893\":{\"denominacion\":\"LAB. HABILIDADES QUIRURGICAS - CBB\",\"descripcion\":\"HAB QUIRURG - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010894\":{\"denominacion\":\"LAB. HABILIDADES COMUNITARIAS - CBB\",\"descripcion\":\"HAB COMUNITA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010895\":{\"denominacion\":\"LAB. HABILIDADES ENFERMERIA - CBB\",\"descripcion\":\"HAB ENFERMER - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010896\":{\"denominacion\":\"GAB. SEGURIDAD E HIGIENE INDUST - CBB\",\"descripcion\":\"SEG E HIG IND - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010897\":{\"denominacion\":\"LAB. INDUSTRIAS PETROQUIMICAS - CBB\",\"descripcion\":\"IND PETROQUI - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010898\":{\"denominacion\":\"LAB. EVALUACIÓN NUTRICIONAL - CBB\",\"descripcion\":\"EV NUTRICIONAL - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010899\":{\"denominacion\":\"SALA DE ESPECIALIDADES - CBB\",\"descripcion\":\"S ESPECIALIDAD - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010900\":{\"denominacion\":\"FABLAB - CBB\",\"descripcion\":\"FABLAB - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010901\":{\"denominacion\":\"NUCLEO ASESORAMIENTO EMPRESARIAL - CBB\",\"descripcion\":\"NAE - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010902\":{\"denominacion\":\"LAB. SERIGRAFÍA - CBB\",\"descripcion\":\"SERIGRAFIA - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10010903\":{\"denominacion\":\"LAB. KINCO - CBB\",\"descripcion\":\"KINCO LAB - CBB\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"100201\":{\"denominacion\":\"DISTRIBUIBLES - LPZ\",\"descripcion\":\"DISTRIBUIBLES - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"\",\"bloqueado\":false},\"10020101\":{\"denominacion\":\"ADMINISTRACIÓN - LPZ\",\"descripcion\":\"ADM. - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020102\":{\"denominacion\":\"MANTENIMIENTO - LPZ\",\"descripcion\":\"MNT. - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020103\":{\"denominacion\":\"INVESTIGACIÓN - LPZ\",\"descripcion\":\"INV. - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020104\":{\"denominacion\":\"EXTENSIÓN - LPZ\",\"descripcion\":\"EXT. - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020105\":{\"denominacion\":\"MARKETING - LPZ\",\"descripcion\":\"MKT. - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020106\":{\"denominacion\":\"CAF - LPZ\",\"descripcion\":\"CAF - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020107\":{\"denominacion\":\"GASTOS ACADÉMICOS - LPZ\",\"descripcion\":\"G.ACAD - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020108\":{\"denominacion\":\"DIRECCIÓN NACIONAL - LPZ\",\"descripcion\":\"DIR.NAL - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020109\":{\"denominacion\":\"SUELDOS ACADÉMICOS - LPZ\",\"descripcion\":\"SUEL. ACAD - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020110\":{\"denominacion\":\"LABORATORIO GENERAL - LPZ\",\"descripcion\":\"LAB. GRAL - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020111\":{\"denominacion\":\"SUELDOS ADMINISTRATIVOS POSTGRADO - LPZ\",\"descripcion\":\"S. ADM. POSTG - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020112\":{\"denominacion\":\"NO APLICABLES A CARRERAS - LPZ\",\"descripcion\":\"NO APLICA CAR - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020113\":{\"denominacion\":\"DISTRIBUIBLE POSTGRADO - LPZ\",\"descripcion\":\"DIST. POSTGR - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020114\":{\"denominacion\":\"ACTIVOS ACADÉMICOS DE USO GENERAL - LPZ\",\"descripcion\":\"ACT ACAD GRAL - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020115\":{\"denominacion\":\"DEPÓSITO TRANSITORIO - LPZ\",\"descripcion\":\"DEP. TRANSITO - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020116\":{\"denominacion\":\"DIRECTORES Y COORDINADORES CARR - LPZ\",\"descripcion\":\"DIREC Y COORD - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020117\":{\"denominacion\":\"SOUVENIRS BOUTIQUE - LPZ\",\"descripcion\":\"SOUVENIR BOUTQ - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020118\":{\"denominacion\":\"PROYECTO HELVETAS - LPZ\",\"descripcion\":\"PRO. HELVETAS - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10020119\":{\"denominacion\":\"PROYECTO CHORRILLOS - LPZ\",\"descripcion\":\"PRO CHORRILLOS - LPZ\",\"departamento\":\"LA PAZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"100202\":{\"denominacion\":\"CIENCIAS DE LA SALUD - LPZ\",\"descripcion\":\"CS.SALUD - LPZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"\",\"bloqueado\":false},\"10020201\":{\"denominacion\":\"MEDICINA - LPZ\",\"descripcion\":\"MED - LPZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020202\":{\"denominacion\":\"ODONTOLOGÍA - LPZ\",\"descripcion\":\"ODO - LPZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020203\":{\"denominacion\":\"BIOQUÍMICA Y FARMACIA - LPZ\",\"descripcion\":\"BYF - LPZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020204\":{\"denominacion\":\"FISIOTERAPIA Y KINESIOLOGÍA - LPZ\",\"descripcion\":\"LFK - LPZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020205\":{\"denominacion\":\"N/AERMERÍA CLINICO QUIRURGICA - LPZ\",\"descripcion\":\"ECQ - LPZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10020206\":{\"denominacion\":\"NUTRICIÓN Y DIETÉTICA - LPZ\",\"descripcion\":\"LND - LPZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100203\":{\"denominacion\":\"CS. EMPRESARIALES Y SOCIALES - LPZ\",\"descripcion\":\"CS.EMP Y SOC - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"\",\"bloqueado\":false},\"10020301\":{\"denominacion\":\"ING. COMERCIAL - LPZ\",\"descripcion\":\"ICO - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020302\":{\"denominacion\":\"ING. COMERCIO INTERNACIONAL - LPZ\",\"descripcion\":\"ICT - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020303\":{\"denominacion\":\"DERECHO Y CIENCIAS JURÍDICAS - LPZ\",\"descripcion\":\"LDE - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020304\":{\"denominacion\":\"COMUNICACIÓN Y M. DIGITALES - LPZ\",\"descripcion\":\"LCM - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020305\":{\"denominacion\":\"ADMINISTRACIÓN DE EMPRESAS - LPZ\",\"descripcion\":\"LAE - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020306\":{\"denominacion\":\"PSICOLOGÍA - LPZ\",\"descripcion\":\"LPS - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020307\":{\"denominacion\":\"CONTADURÍA PÚBLICA - LPZ\",\"descripcion\":\"LCN - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020308\":{\"denominacion\":\"ING. FINANCIERA Y RIESGOS - LPZ\",\"descripcion\":\"IFR - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020309\":{\"denominacion\":\"ING. CIENCIA DATOS E INT. NEGOCIOS - LPZ\",\"descripcion\":\"LCD - LPZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100204\":{\"denominacion\":\"INFORMÁTICA Y ELECTRÓNICA - LPZ\",\"descripcion\":\"INF Y ELEC - LPZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"\",\"bloqueado\":false},\"10020401\":{\"denominacion\":\"ING. BIOMÉDICA - LPZ\",\"descripcion\":\"IBI - LPZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020402\":{\"denominacion\":\"ING. ELECTRÓNICA - LPZ\",\"descripcion\":\"IEL - LPZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020403\":{\"denominacion\":\"ING. ELECTRO Y DE SISTEMAS - LPZ\",\"descripcion\":\"IES - LPZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020404\":{\"denominacion\":\"ING. TELECOMUNICACIONES - LPZ\",\"descripcion\":\"IET - LPZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020405\":{\"denominacion\":\"ING. SISTEMAS INFORMÁTICOS - LPZ\",\"descripcion\":\"ISI - LPZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020406\":{\"denominacion\":\"TEC. SUP. VIDEO JUEGOS - LPZ\",\"descripcion\":\"TSV - LPZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100205\":{\"denominacion\":\"ARQUITECTURA Y TURISMO - LPZ\",\"descripcion\":\"ARQ Y TURIS - LPZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"\",\"bloqueado\":false},\"10020501\":{\"denominacion\":\"ARQUITECTURA Y URBANISMO - LPZ\",\"descripcion\":\"ARQ - LPZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020502\":{\"denominacion\":\"TURISMO Y HOTELERÍA - LPZ\",\"descripcion\":\"LTH - LPZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020503\":{\"denominacion\":\"GASTRONOMÍA - LPZ\",\"descripcion\":\"LGS - LPZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020504\":{\"denominacion\":\"DIS. INTERIORES Y PAISAJIS - LPZ\",\"descripcion\":\"DYP - LPZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020505\":{\"denominacion\":\"DISEÑO GRÁFICO Y COMUNIC. VISUAL - LPZ\",\"descripcion\":\"LDG - LPZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100206\":{\"denominacion\":\"TECNOLOGÍA - LPZ\",\"descripcion\":\"TECNO - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"\",\"bloqueado\":false},\"10020601\":{\"denominacion\":\"N/A. PETROQUÍMICA - LPZ\",\"descripcion\":\"IPQ - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10020602\":{\"denominacion\":\"ING. CIVIL - LPZ\",\"descripcion\":\"ICI - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020603\":{\"denominacion\":\"ING. PETRÓLEO, GAS Y ENERG. - LPZ\",\"descripcion\":\"IPG - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020604\":{\"denominacion\":\"N/A. INDUSTRIAS ALIMENTARIAS - LPZ\",\"descripcion\":\"IIA - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10020605\":{\"denominacion\":\"N/A. AERONÁUTICA - LPZ\",\"descripcion\":\"IAE - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10020606\":{\"denominacion\":\"N/A. ELECTROMECÁNICA - LPZ\",\"descripcion\":\"IEL - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10020607\":{\"denominacion\":\"ING. MECÁNICA Y AUTOM. INDUSTRIAL - LPZ\",\"descripcion\":\"IMT - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020608\":{\"denominacion\":\"ING. INDUSTRIAL Y SISTEMAS - LPZ\",\"descripcion\":\"IIS - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020609\":{\"denominacion\":\"ING. INDUSTRIAL - LPZ\",\"descripcion\":\"IIN - LPZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100207\":{\"denominacion\":\"POSTGRADO - LPZ\",\"descripcion\":\"POSTGRADO - LPZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"\",\"bloqueado\":false},\"10020701\":{\"denominacion\":\"DOCTORADO - LPZ\",\"descripcion\":\"DOCTORADO - LPZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020702\":{\"denominacion\":\"MAESTRÍA - LPZ\",\"descripcion\":\"MAESTRIA - LPZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020703\":{\"denominacion\":\"DIPLOMADO - LPZ\",\"descripcion\":\"DIPLOMADO - LPZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10020704\":{\"denominacion\":\"CURSOS - LPZ\",\"descripcion\":\"CURSOS - LPZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100208\":{\"denominacion\":\"LABORATORIOS - LPZ\",\"descripcion\":\"LABORATORIOS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"\",\"bloqueado\":false},\"10020801\":{\"denominacion\":\"SET DE RADIO - LPZ\",\"descripcion\":\"SET RADIO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020802\":{\"denominacion\":\"SET DE FOTOGRAFIA - LPZ\",\"descripcion\":\"FOTOGRAFIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020803\":{\"denominacion\":\"MUSEO GENERAL - LPZ\",\"descripcion\":\"MUSEO GRAL - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020804\":{\"denominacion\":\"LAB. DE RECURSOS FISICOS - LPZ\",\"descripcion\":\"REC FISI - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020805\":{\"denominacion\":\"LAB. DE ENTRENAMIENTO - LPZ\",\"descripcion\":\"ENTRENAM - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020806\":{\"denominacion\":\"KINESIOLOGIA - LPZ\",\"descripcion\":\"KINESIO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020807\":{\"denominacion\":\"LAB. DE SIMULACIÓN - LPZ\",\"descripcion\":\"SIMULACION - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020808\":{\"denominacion\":\"LAB. DE NEUROLOGÍA - LPZ\",\"descripcion\":\"NEURO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020809\":{\"denominacion\":\"LAB. DE PSICOMOTRICIDAD - LPZ\",\"descripcion\":\"PSICOMO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020810\":{\"denominacion\":\"LAB. HISTOLOGÍA Y PATOLOGÍA - LPZ\",\"descripcion\":\"HIST Y PATO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020811\":{\"denominacion\":\"LAB. GENETICA Y EMBRIOLOGÍA - LPZ\",\"descripcion\":\"GEN Y EMBRIO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020812\":{\"denominacion\":\"LAB. QUÍMICA Y BIOQUÍMICA - LPZ\",\"descripcion\":\"QUI Y BIOQUI - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020813\":{\"denominacion\":\"LAB. DE HEMATOLOGÍA - LPZ\",\"descripcion\":\"HEMATO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020814\":{\"denominacion\":\"SALA DE BALANZAS - LPZ\",\"descripcion\":\"S.BALANZAS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020815\":{\"denominacion\":\"SALA DE EQUIPOS - LPZ\",\"descripcion\":\"S.EQUIPOS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020816\":{\"denominacion\":\"LAB. MICROBIO Y PARASITO - LPZ\",\"descripcion\":\"MICRO Y PARAS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020817\":{\"denominacion\":\"LAB. SEMIOLOGÍA Y FISIOLOGÍA - LPZ\",\"descripcion\":\"SEMI Y FISIO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020818\":{\"denominacion\":\"CIRUGIA - LPZ\",\"descripcion\":\"CIRUGIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020819\":{\"denominacion\":\"ODONTOPEDRIATRÍA - LPZ\",\"descripcion\":\"ODONTOPEDIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020820\":{\"denominacion\":\"LAB. DE PROTESIS - LPZ\",\"descripcion\":\"PROTESIS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020821\":{\"denominacion\":\"QUIROFANO - LPZ\",\"descripcion\":\"QUIROFANO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020822\":{\"denominacion\":\"SALA DE ESTERILIZACIÓN - LPZ\",\"descripcion\":\"S.ESTERILI - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020823\":{\"denominacion\":\"CLINICA (A) - LPZ\",\"descripcion\":\"CLINICA A - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020824\":{\"denominacion\":\"CLINICA (B) - LPZ\",\"descripcion\":\"CLINICA B - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020825\":{\"denominacion\":\"SALA DE ADMICIÓN (CIRUGIA) - LPZ\",\"descripcion\":\"CIR. ADMISION - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020826\":{\"denominacion\":\"SALA RADIOLOGÍA (PERIEPI. PANORA) - LPZ\",\"descripcion\":\"RADIOLOGIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020827\":{\"denominacion\":\"MORFOLOGÍA SALA DE ANATOMIA - LPZ\",\"descripcion\":\"MORFOLOGIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020828\":{\"denominacion\":\"OSTEOTECA - LPZ\",\"descripcion\":\"OSTEOTECA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020829\":{\"denominacion\":\"SALA DE INVESTIGACIÓN - LPZ\",\"descripcion\":\"S. INVESTI - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020830\":{\"denominacion\":\"MUSEO DE ANATOMÍA - LPZ\",\"descripcion\":\"MUSEO ATO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020831\":{\"denominacion\":\"SALA ANATOMÍA Y SIMULACIÓN - LPZ\",\"descripcion\":\"ANATO Y SIM - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020832\":{\"denominacion\":\"DEPOSITO PIEZAS ANATOMICAS - LPZ\",\"descripcion\":\"DEP PIEZ ANAT - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020833\":{\"denominacion\":\"LAB. DE PASTELERIA - LPZ\",\"descripcion\":\"PASTELERIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020834\":{\"denominacion\":\"LAB. DE COCINA - LPZ\",\"descripcion\":\"COCINA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020835\":{\"denominacion\":\"ECONOMATO - LPZ\",\"descripcion\":\"ECONOMATO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020836\":{\"denominacion\":\"LAB. DE PANADERIA - LPZ\",\"descripcion\":\"PANADERIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020837\":{\"denominacion\":\"COCTELERIA - LPZ\",\"descripcion\":\"COCTELERIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020838\":{\"denominacion\":\"SALA DE PEDIATRÍA - LPZ\",\"descripcion\":\"S. PEDIATRIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020839\":{\"denominacion\":\"SALA GINECO. Y OBSTETRICIA - LPZ\",\"descripcion\":\"S. GINE Y OBS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020840\":{\"denominacion\":\"SALA DE SUMINISTROS - LPZ\",\"descripcion\":\"S. SUMINISTRO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020841\":{\"denominacion\":\"SALA DE QUIROFANO - LPZ\",\"descripcion\":\"S. QUIROFANO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020842\":{\"denominacion\":\"SALA DE HOSPITALIZACIÓN - LPZ\",\"descripcion\":\"S. HOSPITAL - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020843\":{\"denominacion\":\"SALA DE TERAPIA INTENSIVA - LPZ\",\"descripcion\":\"S. TER INTENS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020844\":{\"denominacion\":\"SALA DE ATENCIÓN PRIMARIA - LPZ\",\"descripcion\":\"S. ATEN PRIM - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020845\":{\"denominacion\":\"SALA MULTIPROPOSITO - LPZ\",\"descripcion\":\"S. MULTIPROP - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020846\":{\"denominacion\":\"TALLER DE ARQUITECTURA - LPZ\",\"descripcion\":\"T. ARQUITEC - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020847\":{\"denominacion\":\"GB. URBANISMO Y TRAFICO - LPZ\",\"descripcion\":\"G. URB Y TRAF - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020848\":{\"denominacion\":\"INCUBACIÓN - LPZ\",\"descripcion\":\"INCUBACION - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020849\":{\"denominacion\":\"SALA DE JUICIOS ORALES - LPZ\",\"descripcion\":\"S. JUICIOS OR - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020850\":{\"denominacion\":\"NEUROMARKETING - LPZ\",\"descripcion\":\"NEUROMKT - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020851\":{\"denominacion\":\"CONSERVATORIO Y SALA REUNIONES - LPZ\",\"descripcion\":\"CONSER Y REUN - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020852\":{\"denominacion\":\"INNOVACIÓN - LPZ\",\"descripcion\":\"INNOVACION - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020853\":{\"denominacion\":\"SALA DE REUNIONES - LPZ\",\"descripcion\":\"S. REUNION - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020854\":{\"denominacion\":\"CAMARA GESELL - LPZ\",\"descripcion\":\"CAM GESELL - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020855\":{\"denominacion\":\"CENTRO DE COMPUTO - LPZ\",\"descripcion\":\"CEN COMPUTO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020856\":{\"denominacion\":\"LAB. DE BURSATIL - LPZ\",\"descripcion\":\"BURSATIL - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020857\":{\"denominacion\":\"LAB. DISEÑO COMPUTARIZADO - LPZ\",\"descripcion\":\"DIS COMPUTA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020858\":{\"denominacion\":\"LAB. DE FISICA - LPZ\",\"descripcion\":\"FISICA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020859\":{\"denominacion\":\"LAB. ELECTRÓNICA DIGITAL - LPZ\",\"descripcion\":\"ELECT DIGITAL - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020860\":{\"denominacion\":\"LAB. DE ROBOTICA - LPZ\",\"descripcion\":\"ROBOTICA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020861\":{\"denominacion\":\"LAB. TELECOMUNICACIONES - LPZ\",\"descripcion\":\"L. TELECOM - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020862\":{\"denominacion\":\"LAB. ELECTRÓNICA Y BIOMÉDICA - LPZ\",\"descripcion\":\"L. ELEC Y BIO - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020863\":{\"denominacion\":\"LAB. ENERGIAS ALTERNATIVAS - LPZ\",\"descripcion\":\"ENER ALTER - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020864\":{\"denominacion\":\"LAB. LUBRICANTES Y CARBURANTES - LPZ\",\"descripcion\":\"LUBR Y CAR - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020865\":{\"denominacion\":\"LAB. SIMULACIÓN INDUS. INGENI MET - LPZ\",\"descripcion\":\"SIM INDUSTRIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020866\":{\"denominacion\":\"LAB. ANL. INSTRUMEN, GB METRO IND - LPZ\",\"descripcion\":\"A IND ING MET - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020867\":{\"denominacion\":\"GB. SEGURIDAD INDUST. Y TOPOG. - LPZ\",\"descripcion\":\"SEG IND Y TOP - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020868\":{\"denominacion\":\"LAB. DE SANITARIA - LPZ\",\"descripcion\":\"SANITARIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020869\":{\"denominacion\":\"LAB. HIDRAULICA - LPZ\",\"descripcion\":\"HIDRAULICA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020870\":{\"denominacion\":\"LAB. SUELOS, HORMIGONES Y ASFALTO - LPZ\",\"descripcion\":\"SUELOS HORMIG - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020871\":{\"denominacion\":\"LAB. RESISTENCIA MATERIALES - LPZ\",\"descripcion\":\"RESIS MATER - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020872\":{\"denominacion\":\"LAB. PROCESOS INDUSTRIALES - LPZ\",\"descripcion\":\"PROC INDUS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020873\":{\"denominacion\":\"LAB. ANALISIS ESTRUCTURAL - LPZ\",\"descripcion\":\"ANALIS ESTRUC - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020874\":{\"denominacion\":\"LAB. DE DESTILACIÓN - LPZ\",\"descripcion\":\"DESTILACION - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020875\":{\"denominacion\":\"N/A. CONDUCTUAL - LPZ\",\"descripcion\":\"CONDUCTUAL - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020876\":{\"denominacion\":\"N/A. PLANTA DE ALIMENTOS - LPZ\",\"descripcion\":\"PLANTA ALIM. - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020877\":{\"denominacion\":\"N/A. EMBUTIDOS - LPZ\",\"descripcion\":\"EMBUTIDOS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020878\":{\"denominacion\":\"N/A. AUDIO DIGITAL - LPZ\",\"descripcion\":\"AUDIO DIGITAL - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020879\":{\"denominacion\":\"LAB. SET DE TELEVISION - LPZ\",\"descripcion\":\"ST TELEVISION - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020880\":{\"denominacion\":\"N/A QUIMICA ORGANICA E INORGANICA - LPZ\",\"descripcion\":\"QMC ORG/INORG - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020881\":{\"denominacion\":\"N/A. FISICOQUIMICA - LPZ\",\"descripcion\":\"FISICOQUIMICA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020882\":{\"denominacion\":\"N/A. PARASITOLOGIA - LPZ\",\"descripcion\":\"PARASITOLOGIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020883\":{\"denominacion\":\"LAB. ELECTROTENIA - LPZ\",\"descripcion\":\"ELECTROTENIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020884\":{\"denominacion\":\"N/A. POTENCIA - LPZ\",\"descripcion\":\"POTENCIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020885\":{\"denominacion\":\"N/A. MAQUINAS ELECTRICAS - LPZ\",\"descripcion\":\"MAQ. ELECTR. - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020886\":{\"denominacion\":\"LAB. MAQUINAS TERMICAS - LPZ\",\"descripcion\":\"MAQ. TERMIC. - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020887\":{\"denominacion\":\"LAB. PROCESOS - LPZ\",\"descripcion\":\"PROCESOS - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020888\":{\"denominacion\":\"N/A. ANALISIS SENSORIAL - LPZ\",\"descripcion\":\"ANALIS SENSOR - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020889\":{\"denominacion\":\"N/A. OBRAS HIDRAULICAS - LPZ\",\"descripcion\":\"OBRAS HIDRAU - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020890\":{\"denominacion\":\"N/A. AUTOMOTORES - LPZ\",\"descripcion\":\"AUTOMOTORES - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020891\":{\"denominacion\":\"TALLER DE METALMECANICA - LPZ\",\"descripcion\":\"METALMECANICA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020892\":{\"denominacion\":\"BIBLIOTECA - LPZ\",\"descripcion\":\"BIBLIOTECA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020893\":{\"denominacion\":\"N/A. HABILIDADES QUIRURGICAS - LPZ\",\"descripcion\":\"HAB QUIRURG - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020894\":{\"denominacion\":\"N/A. HABILIDADES COMUNITARIAS - LPZ\",\"descripcion\":\"HAB COMUNITA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020895\":{\"denominacion\":\"N/A. HABILIDADES ENFERMERIA - LPZ\",\"descripcion\":\"HAB ENFERMER - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020896\":{\"denominacion\":\"N/A. SEGURIDAD E HIGIENE INDUST - LPZ\",\"descripcion\":\"SEG E HIG IND - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020897\":{\"denominacion\":\"N/A. INDUSTRIAS PETROQUIMICAS - LPZ\",\"descripcion\":\"IND PETROQUI - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10020898\":{\"denominacion\":\"LAB. EVALUACIÓN NUTRICIONAL - LPZ\",\"descripcion\":\"EV NUTRICIONAL - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020899\":{\"denominacion\":\"SALA DE ESPECIALIDADES - LPZ\",\"descripcion\":\"S ESPECIALIDAD - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020900\":{\"denominacion\":\"FABLAB - LPZ\",\"descripcion\":\"FABLAB - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020901\":{\"denominacion\":\"NUCLEO ASESORAMIENTO EMPRESARIAL - LPZ\",\"descripcion\":\"NAE - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020902\":{\"denominacion\":\"LAB. SERIGRAFÍA - LPZ\",\"descripcion\":\"SERIGRAFIA - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10020903\":{\"denominacion\":\"LAB. KINCO - LPZ\",\"descripcion\":\"KINCO LAB - LPZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"100301\":{\"denominacion\":\"DISTRIBUIBLES - SCR\",\"descripcion\":\"DISTRIBUIBLES - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"\",\"bloqueado\":false},\"10030101\":{\"denominacion\":\"ADMINISTRACIÓN - SCR\",\"descripcion\":\"ADM. - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030102\":{\"denominacion\":\"MANTENIMIENTO - SCR\",\"descripcion\":\"MNT. - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030103\":{\"denominacion\":\"INVESTIGACIÓN - SCR\",\"descripcion\":\"INV. - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030104\":{\"denominacion\":\"EXTENSIÓN - SCR\",\"descripcion\":\"EXT. - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030105\":{\"denominacion\":\"MARKETING - SCR\",\"descripcion\":\"MKT. - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030106\":{\"denominacion\":\"CAF - SCR\",\"descripcion\":\"CAF - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030107\":{\"denominacion\":\"GASTOS ACADÉMICOS - SCR\",\"descripcion\":\"G.ACAD - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030108\":{\"denominacion\":\"DIRECCIÓN NACIONAL - SCR\",\"descripcion\":\"DIR.NAL - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030109\":{\"denominacion\":\"SUELDOS ACADÉMICOS - SCR\",\"descripcion\":\"SUEL. ACAD - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030110\":{\"denominacion\":\"LABORATORIO GENERAL - SCR\",\"descripcion\":\"LAB. GRAL - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030111\":{\"denominacion\":\"SUELDOS ADMINISTRATIVOS POSTGRADO - SCR\",\"descripcion\":\"S. ADM. POSTG - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030112\":{\"denominacion\":\"NO APLICABLES A CARRERAS - SCR\",\"descripcion\":\"NO APLICA CAR - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030113\":{\"denominacion\":\"DISTRIBUIBLE POSTGRADO - SCR\",\"descripcion\":\"DIST. POSTGR - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030114\":{\"denominacion\":\"ACTIVOS ACADÉMICOS DE USO GENERAL - SCR\",\"descripcion\":\"ACT ACAD GRAL - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030115\":{\"denominacion\":\"DEPÓSITO TRANSITORIO - SCR\",\"descripcion\":\"DEP. TRANSITO - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030116\":{\"denominacion\":\"DIRECTORES Y COORDINADORES CARR - SCR\",\"descripcion\":\"DIREC Y COORD - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030117\":{\"denominacion\":\"SOUVENIRS BOUTIQUE - SCR\",\"descripcion\":\"SOUVENIR BOUTQ - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030118\":{\"denominacion\":\"PROYECTO HELVETAS - SCR\",\"descripcion\":\"PRO. HELVETAS - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10030119\":{\"denominacion\":\"PROYECTO CHORRILLOS - SCR\",\"descripcion\":\"PRO CHORRILLOS - SCR\",\"departamento\":\"SUCRE\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"100302\":{\"denominacion\":\"CIENCIAS DE LA SALUD - SCR\",\"descripcion\":\"CS.SALUD - SCR\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"\",\"bloqueado\":true},\"10030201\":{\"denominacion\":\"N/AICINA - SCR\",\"descripcion\":\"MED - SCR\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030202\":{\"denominacion\":\"N/ANTOLOGÍA - SCR\",\"descripcion\":\"ODO - SCR\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030203\":{\"denominacion\":\"N/AQUÍMICA Y FARMACIA - SCR\",\"descripcion\":\"BYF - SCR\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030204\":{\"denominacion\":\"N/AIOTERAPIA Y KINESIOLOGÍA - SCR\",\"descripcion\":\"LFK - SCR\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030205\":{\"denominacion\":\"N/AERMERÍA CLINICO QUIRURGICA - SCR\",\"descripcion\":\"ECQ - SCR\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030206\":{\"denominacion\":\"N/ARICIÓN Y DIETÉTICA - SCR\",\"descripcion\":\"LND - SCR\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"100303\":{\"denominacion\":\"CS. EMPRESARIALES Y SOCIALES - SCR\",\"descripcion\":\"CS.EMP Y SOC - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"\",\"bloqueado\":false},\"10030301\":{\"denominacion\":\"ING. COMERCIAL - SCR\",\"descripcion\":\"ICO - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030302\":{\"denominacion\":\"ING. COMERCIO INTERNACIONAL - SCR\",\"descripcion\":\"ICT - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030303\":{\"denominacion\":\"DERECHO Y CIENCIAS JURÍDICAS - SCR\",\"descripcion\":\"LDE - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030304\":{\"denominacion\":\"N/AUNICACIÓN Y M. DIGITALES - SCR\",\"descripcion\":\"LCM - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030305\":{\"denominacion\":\"ADMINISTRACIÓN DE EMPRESAS - SCR\",\"descripcion\":\"LAE - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030306\":{\"denominacion\":\"PSICOLOGÍA - SCR\",\"descripcion\":\"LPS - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030307\":{\"denominacion\":\"CONTADURÍA PÚBLICA - SCR\",\"descripcion\":\"LCN - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030308\":{\"denominacion\":\"ING. FINANCIERA Y RIESGOS - SCR\",\"descripcion\":\"IFR - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030309\":{\"denominacion\":\"ING. CIENCIA DATOS E INT. NEGOCIOS - SCR\",\"descripcion\":\"LCD - SCR\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100304\":{\"denominacion\":\"INFORMÁTICA Y ELECTRÓNICA - SCR\",\"descripcion\":\"INF Y ELEC - SCR\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"\",\"bloqueado\":true},\"10030401\":{\"denominacion\":\"N/A. BIOMÉDICA - SCR\",\"descripcion\":\"IBI - SCR\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030402\":{\"denominacion\":\"N/A. ELECTRÓNICA - SCR\",\"descripcion\":\"IEL - SCR\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030403\":{\"denominacion\":\"N/A. ELECTRO Y DE SISTEMAS - SCR\",\"descripcion\":\"IES - SCR\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030404\":{\"denominacion\":\"N/A. TELECOMUNICACIONES - SCR\",\"descripcion\":\"IET - SCR\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030405\":{\"denominacion\":\"N/A. SISTEMAS INFORMÁTICOS - SCR\",\"descripcion\":\"ISI - SCR\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030406\":{\"denominacion\":\"N/A. SUP. VIDEO JUEGOS - SCR\",\"descripcion\":\"TSV - SCR\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"100305\":{\"denominacion\":\"ARQUITECTURA Y TURISMO - SCR\",\"descripcion\":\"ARQ Y TURIS - SCR\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"\",\"bloqueado\":false},\"10030501\":{\"denominacion\":\"ARQUITECTURA Y URBANISMO - SCR\",\"descripcion\":\"ARQ - SCR\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030502\":{\"denominacion\":\"N/AISMO Y HOTELERÍA - SCR\",\"descripcion\":\"LTH - SCR\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030503\":{\"denominacion\":\"N/ATRONOMÍA - SCR\",\"descripcion\":\"LGS - SCR\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030504\":{\"denominacion\":\"N/A. INTERIORES Y PAISAJIS - SCR\",\"descripcion\":\"DYP - SCR\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030505\":{\"denominacion\":\"DISEÑO GRÁFICO Y COMUNIC. VISUAL - SCR\",\"descripcion\":\"LDG - SCR\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100306\":{\"denominacion\":\"TECNOLOGÍA - SCR\",\"descripcion\":\"TECNO - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"\",\"bloqueado\":false},\"10030601\":{\"denominacion\":\"N/A. PETROQUÍMICA - SCR\",\"descripcion\":\"IPQ - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030602\":{\"denominacion\":\"ING. CIVIL - SCR\",\"descripcion\":\"ICI - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030603\":{\"denominacion\":\"N/A. PETRÓLEO, GAS Y ENERG. - SCR\",\"descripcion\":\"IPG - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030604\":{\"denominacion\":\"N/A. INDUSTRIAS ALIMENTARIAS - SCR\",\"descripcion\":\"IIA - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030605\":{\"denominacion\":\"N/A. AERONÁUTICA - SCR\",\"descripcion\":\"IAE - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030606\":{\"denominacion\":\"N/A. ELECTROMECÁNICA - SCR\",\"descripcion\":\"IEL - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030607\":{\"denominacion\":\"N/A. MECÁNICA Y AUTOM. INDUSTRIAL - SCR\",\"descripcion\":\"IMT - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10030608\":{\"denominacion\":\"ING. INDUSTRIAL Y SISTEMAS - SCR\",\"descripcion\":\"IIS - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030609\":{\"denominacion\":\"N/A. INDUSTRIAL - SCR\",\"descripcion\":\"IIN - SCR\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"100307\":{\"denominacion\":\"POSTGRADO - SCR\",\"descripcion\":\"POSTGRADO - SCR\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"\",\"bloqueado\":true},\"10030701\":{\"denominacion\":\"DOCTORADO - SCR\",\"descripcion\":\"DOCTORADO - SCR\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030702\":{\"denominacion\":\"MAESTRÍA - SCR\",\"descripcion\":\"MAESTRIA - SCR\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030703\":{\"denominacion\":\"DIPLOMADO - SCR\",\"descripcion\":\"DIPLOMADO - SCR\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10030704\":{\"denominacion\":\"CURSOS - SCR\",\"descripcion\":\"CURSOS - SCR\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100308\":{\"denominacion\":\"LABORATORIOS - SCR\",\"descripcion\":\"LABORATORIOS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"\",\"bloqueado\":false},\"10030801\":{\"denominacion\":\"N/A DE RADIO - SCR\",\"descripcion\":\"SET RADIO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030802\":{\"denominacion\":\"N/A DE FOTOGRAFIA - SCR\",\"descripcion\":\"FOTOGRAFIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030803\":{\"denominacion\":\"N/AEO GENERAL - SCR\",\"descripcion\":\"MUSEO GRAL - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030804\":{\"denominacion\":\"N/A. DE RECURSOS FISICOS - SCR\",\"descripcion\":\"REC FISI - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030805\":{\"denominacion\":\"N/A. DE ENTRENAMIENTO - SCR\",\"descripcion\":\"ENTRENAM - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030806\":{\"denominacion\":\"N/AESIOLOGIA - SCR\",\"descripcion\":\"KINESIO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030807\":{\"denominacion\":\"N/A. DE SIMULACIÓN - SCR\",\"descripcion\":\"SIMULACION - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030808\":{\"denominacion\":\"N/A. DE NEUROLOGÍA - SCR\",\"descripcion\":\"NEURO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030809\":{\"denominacion\":\"N/A. DE PSICOMOTRICIDAD - SCR\",\"descripcion\":\"PSICOMO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030810\":{\"denominacion\":\"N/A. HISTOLOGÍA Y PATOLOGÍA - SCR\",\"descripcion\":\"HIST Y PATO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030811\":{\"denominacion\":\"N/A. GENETICA Y EMBRIOLOGÍA - SCR\",\"descripcion\":\"GEN Y EMBRIO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030812\":{\"denominacion\":\"LAB. QUÍMICA Y BIOQUÍMICA - SCR\",\"descripcion\":\"QUI Y BIOQUI - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030813\":{\"denominacion\":\"N/A. DE HEMATOLOGÍA - SCR\",\"descripcion\":\"HEMATO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030814\":{\"denominacion\":\"N/AA DE BALANZAS - SCR\",\"descripcion\":\"S.BALANZAS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030815\":{\"denominacion\":\"N/AA DE EQUIPOS - SCR\",\"descripcion\":\"S.EQUIPOS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030816\":{\"denominacion\":\"N/A. MICROBIO Y PARASITO - SCR\",\"descripcion\":\"MICRO Y PARAS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030817\":{\"denominacion\":\"N/A. SEMIOLOGÍA Y FISIOLOGÍA - SCR\",\"descripcion\":\"SEMI Y FISIO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030818\":{\"denominacion\":\"N/AUGIA - SCR\",\"descripcion\":\"CIRUGIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030819\":{\"denominacion\":\"N/ANTOPEDRIATRÍA - SCR\",\"descripcion\":\"ODONTOPEDIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030820\":{\"denominacion\":\"N/A. DE PROTESIS - SCR\",\"descripcion\":\"PROTESIS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030821\":{\"denominacion\":\"N/AROFANO - SCR\",\"descripcion\":\"QUIROFANO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030822\":{\"denominacion\":\"N/AA DE ESTERILIZACIÓN - SCR\",\"descripcion\":\"S.ESTERILI - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030823\":{\"denominacion\":\"N/ANICA (A) - SCR\",\"descripcion\":\"CLINICA A - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030824\":{\"denominacion\":\"N/ANICA (B) - SCR\",\"descripcion\":\"CLINICA B - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030825\":{\"denominacion\":\"N/AA DE ADMICIÓN (CIRUGIA) - SCR\",\"descripcion\":\"CIR. ADMISION - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030826\":{\"denominacion\":\"N/AA RADIOLOGÍA (PERIEPI. PANORA) - SCR\",\"descripcion\":\"RADIOLOGIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030827\":{\"denominacion\":\"N/AFOLOGÍA SALA DE ANATOMIA - SCR\",\"descripcion\":\"MORFOLOGIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030828\":{\"denominacion\":\"N/AEOTECA - SCR\",\"descripcion\":\"OSTEOTECA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030829\":{\"denominacion\":\"N/AA DE INVESTIGACIÓN - SCR\",\"descripcion\":\"S. INVESTI - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030830\":{\"denominacion\":\"N/AEO DE ANATOMÍA - SCR\",\"descripcion\":\"MUSEO ATO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030831\":{\"denominacion\":\"N/AA ANATOMÍA Y SIMULACIÓN - SCR\",\"descripcion\":\"ANATO Y SIM - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030832\":{\"denominacion\":\"N/AOSITO PIEZAS ANATOMICAS - SCR\",\"descripcion\":\"DEP PIEZ ANAT - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030833\":{\"denominacion\":\"N/A. DE PASTELERIA - SCR\",\"descripcion\":\"PASTELERIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030834\":{\"denominacion\":\"N/A. DE COCINA - SCR\",\"descripcion\":\"COCINA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030835\":{\"denominacion\":\"N/ANOMATO - SCR\",\"descripcion\":\"ECONOMATO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030836\":{\"denominacion\":\"N/A. DE PANADERIA - SCR\",\"descripcion\":\"PANADERIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030837\":{\"denominacion\":\"N/ATELERIA - SCR\",\"descripcion\":\"COCTELERIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030838\":{\"denominacion\":\"N/AA DE PEDIATRÍA - SCR\",\"descripcion\":\"S. PEDIATRIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030839\":{\"denominacion\":\"N/AA GINECO. Y OBSTETRICIA - SCR\",\"descripcion\":\"S. GINE Y OBS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030840\":{\"denominacion\":\"N/AA DE SUMINISTROS - SCR\",\"descripcion\":\"S. SUMINISTRO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030841\":{\"denominacion\":\"N/AA DE QUIROFANO - SCR\",\"descripcion\":\"S. QUIROFANO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030842\":{\"denominacion\":\"N/AA DE HOSPITALIZACIÓN - SCR\",\"descripcion\":\"S. HOSPITAL - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030843\":{\"denominacion\":\"N/AA DE TERAPIA INTENSIVA - SCR\",\"descripcion\":\"S. TER INTENS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030844\":{\"denominacion\":\"N/AA DE ATENCIÓN PRIMARIA - SCR\",\"descripcion\":\"S. ATEN PRIM - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030845\":{\"denominacion\":\"N/AA MULTIPROPOSITO - SCR\",\"descripcion\":\"S. MULTIPROP - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030846\":{\"denominacion\":\"TALLER DE ARQUITECTURA - SCR\",\"descripcion\":\"T. ARQUITEC - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030847\":{\"denominacion\":\"GB. URBANISMO Y TRAFICO - SCR\",\"descripcion\":\"G. URB Y TRAF - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030848\":{\"denominacion\":\"N/AUBACIÓN - SCR\",\"descripcion\":\"INCUBACION - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030849\":{\"denominacion\":\"SALA DE JUICIOS ORALES - SCR\",\"descripcion\":\"S. JUICIOS OR - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030850\":{\"denominacion\":\"N/AROMARKETING - SCR\",\"descripcion\":\"NEUROMKT - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030851\":{\"denominacion\":\"N/ASERVATORIO Y SALA REUNIONES - SCR\",\"descripcion\":\"CONSER Y REUN - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030852\":{\"denominacion\":\"N/AOVACIÓN - SCR\",\"descripcion\":\"INNOVACION - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030853\":{\"denominacion\":\"N/AA DE REUNIONES - SCR\",\"descripcion\":\"S. REUNION - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030854\":{\"denominacion\":\"CAMARA GESELL - SCR\",\"descripcion\":\"CAM GESELL - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030855\":{\"denominacion\":\"CENTRO DE COMPUTO - SCR\",\"descripcion\":\"CEN COMPUTO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030856\":{\"denominacion\":\"N/A. DE BURSATIL - SCR\",\"descripcion\":\"BURSATIL - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030857\":{\"denominacion\":\"N/A. DISEÑO COMPUTARIZADO - SCR\",\"descripcion\":\"DIS COMPUTA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030858\":{\"denominacion\":\"LAB. DE FISICA - SCR\",\"descripcion\":\"FISICA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030859\":{\"denominacion\":\"LAB. ELECTRÓNICA DIGITAL - SCR\",\"descripcion\":\"ELECT DIGITAL - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030860\":{\"denominacion\":\"N/A. DE ROBOTICA - SCR\",\"descripcion\":\"ROBOTICA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030861\":{\"denominacion\":\"N/A. TELECOMUNICACIONES - SCR\",\"descripcion\":\"L. TELECOM - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030862\":{\"denominacion\":\"N/A. ELECTRÓNICA Y BIOMÉDICA - SCR\",\"descripcion\":\"L. ELEC Y BIO - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030863\":{\"denominacion\":\"N/A. ENERGIAS ALTERNATIVAS - SCR\",\"descripcion\":\"ENER ALTER - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030864\":{\"denominacion\":\"N/A. LUBRICANTES Y CARBURANTES - SCR\",\"descripcion\":\"LUBR Y CAR - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030865\":{\"denominacion\":\"N/A. SIMULACIÓN INDUS. INGENI MET - SCR\",\"descripcion\":\"SIM INDUSTRIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030866\":{\"denominacion\":\"N/A. ANL. INSTRUMEN, GB METRO IND - SCR\",\"descripcion\":\"A IND ING MET - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030867\":{\"denominacion\":\"GB. SEGURIDAD INDUST. Y TOPOG. - SCR\",\"descripcion\":\"SEG IND Y TOP - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030868\":{\"denominacion\":\"LAB. DE SANITARIA - SCR\",\"descripcion\":\"SANITARIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030869\":{\"denominacion\":\"LAB. HIDRAULICA - SCR\",\"descripcion\":\"HIDRAULICA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030870\":{\"denominacion\":\"LAB. SUELOS, HORMIGONES Y ASFALTO - SCR\",\"descripcion\":\"SUELOS HORMIG - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030871\":{\"denominacion\":\"N/A. RESISTENCIA MATERIALES - SCR\",\"descripcion\":\"RESIS MATER - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030872\":{\"denominacion\":\"N/A. PROCESOS INDUSTRIALES - SCR\",\"descripcion\":\"PROC INDUS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030873\":{\"denominacion\":\"N/A. ANALISIS ESTRUCTURAL - SCR\",\"descripcion\":\"ANALIS ESTRUC - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030874\":{\"denominacion\":\"N/A. DE DESTILACIÓN - SCR\",\"descripcion\":\"DESTILACION - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030875\":{\"denominacion\":\"LAB. CONDUCTUAL - SCR\",\"descripcion\":\"CONDUCTUAL - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030876\":{\"denominacion\":\"N/A. PLANTA DE ALIMENTOS - SCR\",\"descripcion\":\"PLANTA ALIM. - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030877\":{\"denominacion\":\"N/A. EMBUTIDOS - SCR\",\"descripcion\":\"EMBUTIDOS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030878\":{\"denominacion\":\"N/A. AUDIO DIGITAL - SCR\",\"descripcion\":\"AUDIO DIGITAL - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030879\":{\"denominacion\":\"N/A. SET DE TELEVISION - SCR\",\"descripcion\":\"ST TELEVISION - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030880\":{\"denominacion\":\"N/A QUIMICA ORGANICA E INORGANICA - SCR\",\"descripcion\":\"QMC ORG/INORG - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030881\":{\"denominacion\":\"N/A. FISICOQUIMICA - SCR\",\"descripcion\":\"FISICOQUIMICA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030882\":{\"denominacion\":\"N/A. PARASITOLOGIA - SCR\",\"descripcion\":\"PARASITOLOGIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030883\":{\"denominacion\":\"N/A. ELECTROTENIA - SCR\",\"descripcion\":\"ELECTROTENIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030884\":{\"denominacion\":\"N/A. POTENCIA - SCR\",\"descripcion\":\"POTENCIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030885\":{\"denominacion\":\"N/A. MAQUINAS ELECTRICAS - SCR\",\"descripcion\":\"MAQ. ELECTR. - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030886\":{\"denominacion\":\"N/A. MAQUINAS TERMICAS - SCR\",\"descripcion\":\"MAQ. TERMIC. - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030887\":{\"denominacion\":\"N/A. PROCESOS - SCR\",\"descripcion\":\"PROCESOS - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030888\":{\"denominacion\":\"N/A. ANALISIS SENSORIAL - SCR\",\"descripcion\":\"ANALIS SENSOR - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030889\":{\"denominacion\":\"N/A. OBRAS HIDRAULICAS - SCR\",\"descripcion\":\"OBRAS HIDRAU - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030890\":{\"denominacion\":\"N/A. AUTOMOTORES - SCR\",\"descripcion\":\"AUTOMOTORES - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030891\":{\"denominacion\":\"N/ALER DE METALMECANICA - SCR\",\"descripcion\":\"METALMECANICA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030892\":{\"denominacion\":\"BIBLIOTECA - SCR\",\"descripcion\":\"BIBLIOTECA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030893\":{\"denominacion\":\"N/A. HABILIDADES QUIRURGICAS - SCR\",\"descripcion\":\"HAB QUIRURG - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030894\":{\"denominacion\":\"N/A. HABILIDADES COMUNITARIAS - SCR\",\"descripcion\":\"HAB COMUNITA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030895\":{\"denominacion\":\"N/A. HABILIDADES ENFERMERIA - SCR\",\"descripcion\":\"HAB ENFERMER - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030896\":{\"denominacion\":\"N/A. SEGURIDAD E HIGIENE INDUST - SCR\",\"descripcion\":\"SEG E HIG IND - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030897\":{\"denominacion\":\"N/A. INDUSTRIAS PETROQUIMICAS - SCR\",\"descripcion\":\"IND PETROQUI - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030898\":{\"denominacion\":\"LAB. EVALUACIÓN NUTRICIONAL - SCR\",\"descripcion\":\"EV NUTRICIONAL - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030899\":{\"denominacion\":\"SALA DE ESPECIALIDADES - SCR\",\"descripcion\":\"S ESPECIALIDAD - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030900\":{\"denominacion\":\"N/ALAB - SCR\",\"descripcion\":\"FABLAB - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10030901\":{\"denominacion\":\"NUCLEO ASESORAMIENTO EMPRESARIAL - SCR\",\"descripcion\":\"NAE - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030902\":{\"denominacion\":\"LAB. SERIGRAFÍA - SCR\",\"descripcion\":\"SERIGRAFIA - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10030903\":{\"denominacion\":\"LAB. KINCO - SCR\",\"descripcion\":\"KINCO LAB - SCR\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"100401\":{\"denominacion\":\"DISTRIBUIBLES - TDD\",\"descripcion\":\"DISTRIBUIBLES - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"\",\"bloqueado\":false},\"10040101\":{\"denominacion\":\"ADMINISTRACIÓN - TDD\",\"descripcion\":\"ADM. - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040102\":{\"denominacion\":\"MANTENIMIENTO - TDD\",\"descripcion\":\"MNT. - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040103\":{\"denominacion\":\"INVESTIGACIÓN - TDD\",\"descripcion\":\"INV. - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040104\":{\"denominacion\":\"EXTENSIÓN - TDD\",\"descripcion\":\"EXT. - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040105\":{\"denominacion\":\"MARKETING - TDD\",\"descripcion\":\"MKT. - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040106\":{\"denominacion\":\"CAF - TDD\",\"descripcion\":\"CAF - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040107\":{\"denominacion\":\"GASTOS ACADÉMICOS - TDD\",\"descripcion\":\"G.ACAD - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040108\":{\"denominacion\":\"DIRECCIÓN NACIONAL - TDD\",\"descripcion\":\"DIR.NAL - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040109\":{\"denominacion\":\"SUELDOS ACADÉMICOS - TDD\",\"descripcion\":\"SUEL. ACAD - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040110\":{\"denominacion\":\"LABORATORIO GENERAL - TDD\",\"descripcion\":\"LAB. GRAL - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040111\":{\"denominacion\":\"SUELDOS ADMINISTRATIVOS POSTGRADO - TDD\",\"descripcion\":\"S. ADM. POSTG - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040112\":{\"denominacion\":\"NO APLICABLES A CARRERAS - TDD\",\"descripcion\":\"NO APLICA CAR - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040113\":{\"denominacion\":\"DISTRIBUIBLE POSTGRADO - TDD\",\"descripcion\":\"DIST. POSTGR - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040114\":{\"denominacion\":\"ACTIVOS ACADÉMICOS DE USO GENERAL - TDD\",\"descripcion\":\"ACT ACAD GRAL - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040115\":{\"denominacion\":\"DEPÓSITO TRANSITORIO - TDD\",\"descripcion\":\"DEP. TRANSITO - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040116\":{\"denominacion\":\"DIRECTORES Y COORDINADORES CARR - TDD\",\"descripcion\":\"DIREC Y COORD - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040117\":{\"denominacion\":\"SOUVENIRS BOUTIQUE - TDD\",\"descripcion\":\"SOUVENIR BOUTQ - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040118\":{\"denominacion\":\"PROYECTO HELVETAS - TDD\",\"descripcion\":\"PRO. HELVETAS - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10040119\":{\"denominacion\":\"PROYECTO CHORRILLOS - TDD\",\"descripcion\":\"PRO CHORRILLOS - TDD\",\"departamento\":\"TRINIDAD\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"100402\":{\"denominacion\":\"CIENCIAS DE LA SALUD - TDD\",\"descripcion\":\"CS.SALUD - TDD\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"\",\"bloqueado\":true},\"10040201\":{\"denominacion\":\"N/AICINA - TDD\",\"descripcion\":\"MED - TDD\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040202\":{\"denominacion\":\"N/ANTOLOGÍA - TDD\",\"descripcion\":\"ODO - TDD\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040203\":{\"denominacion\":\"N/AQUÍMICA Y FARMACIA - TDD\",\"descripcion\":\"BYF - TDD\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040204\":{\"denominacion\":\"N/AIOTERAPIA Y KINESIOLOGÍA - TDD\",\"descripcion\":\"LFK - TDD\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040205\":{\"denominacion\":\"N/AERMERÍA CLINICO QUIRURGICA - TDD\",\"descripcion\":\"ECQ - TDD\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040206\":{\"denominacion\":\"N/ARICIÓN Y DIETÉTICA - TDD\",\"descripcion\":\"LND - TDD\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"100403\":{\"denominacion\":\"CS. EMPRESARIALES Y SOCIALES - TDD\",\"descripcion\":\"CS.EMP Y SOC - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"\",\"bloqueado\":false},\"10040301\":{\"denominacion\":\"ING. COMERCIAL - TDD\",\"descripcion\":\"ICO - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040302\":{\"denominacion\":\"ING. COMERCIO INTERNACIONAL - TDD\",\"descripcion\":\"ICT - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040303\":{\"denominacion\":\"DERECHO Y CIENCIAS JURÍDICAS - TDD\",\"descripcion\":\"LDE - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040304\":{\"denominacion\":\"N/AUNICACIÓN Y M. DIGITALES - TDD\",\"descripcion\":\"LCM - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040305\":{\"denominacion\":\"ADMINISTRACIÓN DE EMPRESAS - TDD\",\"descripcion\":\"LAE - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040306\":{\"denominacion\":\"N/ACOLOGÍA - TDD\",\"descripcion\":\"LPS - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040307\":{\"denominacion\":\"CONTADURÍA PÚBLICA - TDD\",\"descripcion\":\"LCN - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040308\":{\"denominacion\":\"N/A. FINANCIERA Y RIESGOS - TDD\",\"descripcion\":\"IFR - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040309\":{\"denominacion\":\"N/A. CIENCIA DATOS E INT. NEGOCIOS - TDD\",\"descripcion\":\"LCD - TDD\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"100404\":{\"denominacion\":\"INFORMÁTICA Y ELECTRÓNICA - TDD\",\"descripcion\":\"INF Y ELEC - TDD\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"\",\"bloqueado\":true},\"10040401\":{\"denominacion\":\"N/A. BIOMÉDICA - TDD\",\"descripcion\":\"IBI - TDD\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040402\":{\"denominacion\":\"N/A. ELECTRÓNICA - TDD\",\"descripcion\":\"IEL - TDD\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040403\":{\"denominacion\":\"N/A. ELECTRO Y DE SISTEMAS - TDD\",\"descripcion\":\"IES - TDD\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040404\":{\"denominacion\":\"N/A. TELECOMUNICACIONES - TDD\",\"descripcion\":\"IET - TDD\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040405\":{\"denominacion\":\"ING. SISTEMAS INFORMÁTICOS - TDD\",\"descripcion\":\"ISI - TDD\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040406\":{\"denominacion\":\"N/A. SUP. VIDEO JUEGOS - TDD\",\"descripcion\":\"TSV - TDD\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"100405\":{\"denominacion\":\"ARQUITECTURA Y TURISMO - TDD\",\"descripcion\":\"ARQ Y TURIS - TDD\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"\",\"bloqueado\":false},\"10040501\":{\"denominacion\":\"ARQUITECTURA Y URBANISMO - TDD\",\"descripcion\":\"ARQ - TDD\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040502\":{\"denominacion\":\"N/AISMO Y HOTELERÍA - TDD\",\"descripcion\":\"LTH - TDD\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040503\":{\"denominacion\":\"N/ATRONOMÍA - TDD\",\"descripcion\":\"LGS - TDD\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040504\":{\"denominacion\":\"N/A. INTERIORES Y PAISAJIS - TDD\",\"descripcion\":\"DYP - TDD\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040505\":{\"denominacion\":\"DISEÑO GRÁFICO Y COMUNIC. VISUAL - TDD\",\"descripcion\":\"LDG - TDD\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100406\":{\"denominacion\":\"TECNOLOGÍA - TDD\",\"descripcion\":\"TECNO - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"\",\"bloqueado\":true},\"10040601\":{\"denominacion\":\"N/A. PETROQUÍMICA - TDD\",\"descripcion\":\"IPQ - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040602\":{\"denominacion\":\"ING. CIVIL - TDD\",\"descripcion\":\"ICI - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040603\":{\"denominacion\":\"N/A. PETRÓLEO, GAS Y ENERG. - TDD\",\"descripcion\":\"IPG - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040604\":{\"denominacion\":\"N/A. INDUSTRIAS ALIMENTARIAS - TDD\",\"descripcion\":\"IIA - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040605\":{\"denominacion\":\"N/A. AERONÁUTICA - TDD\",\"descripcion\":\"IAE - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040606\":{\"denominacion\":\"N/A. ELECTROMECÁNICA - TDD\",\"descripcion\":\"IEL - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040607\":{\"denominacion\":\"N/A. MECÁNICA Y AUTOM. INDUSTRIAL - TDD\",\"descripcion\":\"IMT - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040608\":{\"denominacion\":\"N/A. INDUSTRIAL Y SISTEMAS - TDD\",\"descripcion\":\"IIS - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10040609\":{\"denominacion\":\"N/A. INDUSTRIAL - TDD\",\"descripcion\":\"IIN - TDD\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"100407\":{\"denominacion\":\"POSTGRADO - TDD\",\"descripcion\":\"POSTGRADO - TDD\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"\",\"bloqueado\":false},\"10040701\":{\"denominacion\":\"DOCTORADO - TDD\",\"descripcion\":\"DOCTORADO - TDD\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040702\":{\"denominacion\":\"MAESTRÍA - TDD\",\"descripcion\":\"MAESTRIA - TDD\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040703\":{\"denominacion\":\"DIPLOMADO - TDD\",\"descripcion\":\"DIPLOMADO - TDD\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10040704\":{\"denominacion\":\"CURSOS - TDD\",\"descripcion\":\"CURSOS - TDD\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100408\":{\"denominacion\":\"LABORATORIOS - TDD\",\"descripcion\":\"LABORATORIOS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"\",\"bloqueado\":false},\"10040801\":{\"denominacion\":\"LAB DE RADIO - TDD\",\"descripcion\":\"SET RADIO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040802\":{\"denominacion\":\"N/A DE FOTOGRAFIA - TDD\",\"descripcion\":\"FOTOGRAFIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040803\":{\"denominacion\":\"N/AEO GENERAL - TDD\",\"descripcion\":\"MUSEO GRAL - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040804\":{\"denominacion\":\"N/A. DE RECURSOS FISICOS - TDD\",\"descripcion\":\"REC FISI - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040805\":{\"denominacion\":\"N/A. DE ENTRENAMIENTO - TDD\",\"descripcion\":\"ENTRENAM - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040806\":{\"denominacion\":\"N/AESIOLOGIA - TDD\",\"descripcion\":\"KINESIO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040807\":{\"denominacion\":\"N/A. DE SIMULACIÓN - TDD\",\"descripcion\":\"SIMULACION - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040808\":{\"denominacion\":\"N/A. DE NEUROLOGÍA - TDD\",\"descripcion\":\"NEURO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040809\":{\"denominacion\":\"N/A. DE PSICOMOTRICIDAD - TDD\",\"descripcion\":\"PSICOMO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040810\":{\"denominacion\":\"N/A. HISTOLOGÍA Y PATOLOGÍA - TDD\",\"descripcion\":\"HIST Y PATO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040811\":{\"denominacion\":\"N/A. GENETICA Y EMBRIOLOGÍA - TDD\",\"descripcion\":\"GEN Y EMBRIO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040812\":{\"denominacion\":\"N/A. QUÍMICA Y BIOQUÍMICA - TDD\",\"descripcion\":\"QUI Y BIOQUI - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040813\":{\"denominacion\":\"N/A. DE HEMATOLOGÍA - TDD\",\"descripcion\":\"HEMATO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040814\":{\"denominacion\":\"N/AA DE BALANZAS - TDD\",\"descripcion\":\"S.BALANZAS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040815\":{\"denominacion\":\"N/AA DE EQUIPOS - TDD\",\"descripcion\":\"S.EQUIPOS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040816\":{\"denominacion\":\"N/A. MICROBIO Y PARASITO - TDD\",\"descripcion\":\"MICRO Y PARAS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040817\":{\"denominacion\":\"N/A. SEMIOLOGÍA Y FISIOLOGÍA - TDD\",\"descripcion\":\"SEMI Y FISIO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040818\":{\"denominacion\":\"N/AUGIA - TDD\",\"descripcion\":\"CIRUGIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040819\":{\"denominacion\":\"N/ANTOPEDRIATRÍA - TDD\",\"descripcion\":\"ODONTOPEDIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040820\":{\"denominacion\":\"N/A. DE PROTESIS - TDD\",\"descripcion\":\"PROTESIS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040821\":{\"denominacion\":\"N/AROFANO - TDD\",\"descripcion\":\"QUIROFANO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040822\":{\"denominacion\":\"N/AA DE ESTERILIZACIÓN - TDD\",\"descripcion\":\"S.ESTERILI - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040823\":{\"denominacion\":\"N/ANICA (A) - TDD\",\"descripcion\":\"CLINICA A - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040824\":{\"denominacion\":\"N/ANICA (B) - TDD\",\"descripcion\":\"CLINICA B - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040825\":{\"denominacion\":\"N/AA DE ADMICIÓN (CIRUGIA) - TDD\",\"descripcion\":\"CIR. ADMISION - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040826\":{\"denominacion\":\"N/AA RADIOLOGÍA (PERIEPI. PANORA) - TDD\",\"descripcion\":\"RADIOLOGIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040827\":{\"denominacion\":\"N/AFOLOGÍA N/AA DE ANATOMIA - TDD\",\"descripcion\":\"MORFOLOGIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040828\":{\"denominacion\":\"N/AEOTECA - TDD\",\"descripcion\":\"OSTEOTECA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040829\":{\"denominacion\":\"N/AA DE INVESTIGACIÓN - TDD\",\"descripcion\":\"S. INVESTI - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040830\":{\"denominacion\":\"N/AEO DE ANATOMÍA - TDD\",\"descripcion\":\"MUSEO ATO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040831\":{\"denominacion\":\"N/AA ANATOMÍA Y SIMULACIÓN - TDD\",\"descripcion\":\"ANATO Y SIM - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040832\":{\"denominacion\":\"N/AOSITO PIEZAS ANATOMICAS - TDD\",\"descripcion\":\"DEP PIEZ ANAT - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040833\":{\"denominacion\":\"N/A. DE PASTELERIA - TDD\",\"descripcion\":\"PASTELERIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040834\":{\"denominacion\":\"N/A. DE COCINA - TDD\",\"descripcion\":\"COCINA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040835\":{\"denominacion\":\"N/ANOMATO - TDD\",\"descripcion\":\"ECONOMATO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040836\":{\"denominacion\":\"N/A. DE PANADERIA - TDD\",\"descripcion\":\"PANADERIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040837\":{\"denominacion\":\"N/ATELERIA - TDD\",\"descripcion\":\"COCTELERIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040838\":{\"denominacion\":\"N/AA DE PEDIATRÍA - TDD\",\"descripcion\":\"S. PEDIATRIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040839\":{\"denominacion\":\"N/AA GINECO. Y OBSTETRICIA - TDD\",\"descripcion\":\"S. GINE Y OBS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040840\":{\"denominacion\":\"N/AA DE SUMINISTROS - TDD\",\"descripcion\":\"S. SUMINISTRO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040841\":{\"denominacion\":\"N/AA DE QUIROFANO - TDD\",\"descripcion\":\"S. QUIROFANO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040842\":{\"denominacion\":\"N/AA DE HOSPITALIZACIÓN - TDD\",\"descripcion\":\"S. HOSPITAL - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040843\":{\"denominacion\":\"N/AA DE TERAPIA INTENSIVA - TDD\",\"descripcion\":\"S. TER INTENS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040844\":{\"denominacion\":\"N/AA DE ATENCIÓN PRIMARIA - TDD\",\"descripcion\":\"S. ATEN PRIM - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040845\":{\"denominacion\":\"N/AA MULTIPROPOSITO - TDD\",\"descripcion\":\"S. MULTIPROP - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040846\":{\"denominacion\":\"N/ALER DE ARQUITECTURA - TDD\",\"descripcion\":\"T. ARQUITEC - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040847\":{\"denominacion\":\"N/A URBANISMO Y TRAFICO - TDD\",\"descripcion\":\"G. URB Y TRAF - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040848\":{\"denominacion\":\"N/AUBACIÓN - TDD\",\"descripcion\":\"INCUBACION - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040849\":{\"denominacion\":\"N/AA DE JUICIOS ORALES - TDD\",\"descripcion\":\"S. JUICIOS OR - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040850\":{\"denominacion\":\"N/AROMARKETING - TDD\",\"descripcion\":\"NEUROMKT - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040851\":{\"denominacion\":\"N/ASERVATORIO Y N/AA REUNIONES - TDD\",\"descripcion\":\"CONSER Y REUN - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040852\":{\"denominacion\":\"N/AOVACIÓN - TDD\",\"descripcion\":\"INNOVACION - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040853\":{\"denominacion\":\"N/AA DE REUNIONES - TDD\",\"descripcion\":\"S. REUNION - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040854\":{\"denominacion\":\"N/AARA GESELL - TDD\",\"descripcion\":\"CAM GESELL - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040855\":{\"denominacion\":\"CENTRO DE COMPUTO - TDD\",\"descripcion\":\"CEN COMPUTO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040856\":{\"denominacion\":\"N/A. DE BURSATIL - TDD\",\"descripcion\":\"BURSATIL - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040857\":{\"denominacion\":\"N/A. DISEÑO COMPUTARIZADO - TDD\",\"descripcion\":\"DIS COMPUTA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040858\":{\"denominacion\":\"LAB. DE FISICA - TDD\",\"descripcion\":\"FISICA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040859\":{\"denominacion\":\"N/A. ELECTRÓNICA DIGITAL - TDD\",\"descripcion\":\"ELECT DIGITAL - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040860\":{\"denominacion\":\"N/A. DE ROBOTICA - TDD\",\"descripcion\":\"ROBOTICA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040861\":{\"denominacion\":\"N/A. TELECOMUNICACIONES - TDD\",\"descripcion\":\"L. TELECOM - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040862\":{\"denominacion\":\"N/A. ELECTRÓNICA Y BIOMÉDICA - TDD\",\"descripcion\":\"L. ELEC Y BIO - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040863\":{\"denominacion\":\"N/A. ENERGIAS ALTERNATIVAS - TDD\",\"descripcion\":\"ENER ALTER - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040864\":{\"denominacion\":\"N/A. LUBRICANTES Y CARBURANTES - TDD\",\"descripcion\":\"LUBR Y CAR - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040865\":{\"denominacion\":\"N/A. SIMULACIÓN INDUS. INGENI MET - TDD\",\"descripcion\":\"SIM INDUSTRIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040866\":{\"denominacion\":\"N/A. ANL. INSTRUMEN, GB METRO IND - TDD\",\"descripcion\":\"A IND ING MET - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040867\":{\"denominacion\":\"GB. SEGURIDAD INDUST. Y TOPOG. - TDD\",\"descripcion\":\"SEG IND Y TOP - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040868\":{\"denominacion\":\"N/A. DE SANITARIA - TDD\",\"descripcion\":\"SANITARIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040869\":{\"denominacion\":\"LAB. HIDRAULICA - TDD\",\"descripcion\":\"HIDRAULICA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040870\":{\"denominacion\":\"N/A. SUELOS, HORMIGONES Y ASFALTO - TDD\",\"descripcion\":\"SUELOS HORMIG - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040871\":{\"denominacion\":\"N/A. RESISTENCIA MATERIALES - TDD\",\"descripcion\":\"RESIS MATER - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040872\":{\"denominacion\":\"N/A. PROCESOS INDUSTRIALES - TDD\",\"descripcion\":\"PROC INDUS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040873\":{\"denominacion\":\"LAB. ANALISIS ESTRUCTURAL - TDD\",\"descripcion\":\"ANALIS ESTRUC - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040874\":{\"denominacion\":\"N/A. DE DESTILACIÓN - TDD\",\"descripcion\":\"DESTILACION - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040875\":{\"denominacion\":\"N/A. CONDUCTUAL - TDD\",\"descripcion\":\"CONDUCTUAL - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040876\":{\"denominacion\":\"N/A. PLANTA DE ALIMENTOS - TDD\",\"descripcion\":\"PLANTA ALIM. - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040877\":{\"denominacion\":\"N/A. EMBUTIDOS - TDD\",\"descripcion\":\"EMBUTIDOS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040878\":{\"denominacion\":\"N/A. AUDIO DIGITAL - TDD\",\"descripcion\":\"AUDIO DIGITAL - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040879\":{\"denominacion\":\"N/A. SET DE TELEVISION - TDD\",\"descripcion\":\"ST TELEVISION - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040880\":{\"denominacion\":\"N/A QUIMICA ORGANICA E INORGANICA - TDD\",\"descripcion\":\"QMC ORG/INORG - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040881\":{\"denominacion\":\"N/A. FISICOQUIMICA - TDD\",\"descripcion\":\"FISICOQUIMICA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040882\":{\"denominacion\":\"N/A. PARASITOLOGIA - TDD\",\"descripcion\":\"PARASITOLOGIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040883\":{\"denominacion\":\"N/A. ELECTROTENIA - TDD\",\"descripcion\":\"ELECTROTENIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040884\":{\"denominacion\":\"N/A. POTENCIA - TDD\",\"descripcion\":\"POTENCIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040885\":{\"denominacion\":\"N/A. MAQUINAS ELECTRICAS - TDD\",\"descripcion\":\"MAQ. ELECTR. - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040886\":{\"denominacion\":\"N/A. MAQUINAS TERMICAS - TDD\",\"descripcion\":\"MAQ. TERMIC. - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040887\":{\"denominacion\":\"N/A. PROCESOS - TDD\",\"descripcion\":\"PROCESOS - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040888\":{\"denominacion\":\"N/A. ANALISIS SENSORIAL - TDD\",\"descripcion\":\"ANALIS SENSOR - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040889\":{\"denominacion\":\"N/A. OBRAS HIDRAULICAS - TDD\",\"descripcion\":\"OBRAS HIDRAU - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040890\":{\"denominacion\":\"N/A. AUTOMOTORES - TDD\",\"descripcion\":\"AUTOMOTORES - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040891\":{\"denominacion\":\"N/ALER DE METALMECANICA - TDD\",\"descripcion\":\"METALMECANICA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040892\":{\"denominacion\":\"BIBLIOTECA - TDD\",\"descripcion\":\"BIBLIOTECA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040893\":{\"denominacion\":\"N/A. HABILIDADES QUIRURGICAS - TDD\",\"descripcion\":\"HAB QUIRURG - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040894\":{\"denominacion\":\"N/A. HABILIDADES COMUNITARIAS - TDD\",\"descripcion\":\"HAB COMUNITA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040895\":{\"denominacion\":\"N/A. HABILIDADES ENFERMERIA - TDD\",\"descripcion\":\"HAB ENFERMER - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040896\":{\"denominacion\":\"N/A. SEGURIDAD E HIGIENE INDUST - TDD\",\"descripcion\":\"SEG E HIG IND - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040897\":{\"denominacion\":\"N/A. INDUSTRIAS PETROQUIMICAS - TDD\",\"descripcion\":\"IND PETROQUI - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040898\":{\"denominacion\":\"LAB. EVALUACIÓN NUTRICIONAL - TDD\",\"descripcion\":\"EV NUTRICIONAL - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040899\":{\"denominacion\":\"SALA DE ESPECIALIDADES - TDD\",\"descripcion\":\"S ESPECIALIDAD - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040900\":{\"denominacion\":\"N/ALAB - TDD\",\"descripcion\":\"FABLAB - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10040901\":{\"denominacion\":\"NUCLEO ASESORAMIENTO EMPRESARIAL - TDD\",\"descripcion\":\"NAE - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040902\":{\"denominacion\":\"LAB. SERIGRAFÍA - TDD\",\"descripcion\":\"SERIGRAFIA - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10040903\":{\"denominacion\":\"LAB. KINCO - TDD\",\"descripcion\":\"KINCO LAB - TDD\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"100501\":{\"denominacion\":\"DISTRIBUIBLES - SCZ\",\"descripcion\":\"DISTRIBUIBLES - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"\",\"bloqueado\":false},\"10050101\":{\"denominacion\":\"ADMINISTRACIÓN - SCZ\",\"descripcion\":\"ADM. - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050102\":{\"denominacion\":\"MANTENIMIENTO - SCZ\",\"descripcion\":\"MNT. - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050103\":{\"denominacion\":\"INVESTIGACIÓN - SCZ\",\"descripcion\":\"INV. - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050104\":{\"denominacion\":\"EXTENSIÓN - SCZ\",\"descripcion\":\"EXT. - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050105\":{\"denominacion\":\"MARKETING - SCZ\",\"descripcion\":\"MKT. - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050106\":{\"denominacion\":\"CAF - SCZ\",\"descripcion\":\"CAF - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050107\":{\"denominacion\":\"GASTOS ACADÉMICOS - SCZ\",\"descripcion\":\"G.ACAD - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050108\":{\"denominacion\":\"DIRECCIÓN NACIONAL - SCZ\",\"descripcion\":\"DIR.NAL - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050109\":{\"denominacion\":\"SUELDOS ACADÉMICOS - SCZ\",\"descripcion\":\"SUEL. ACAD - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050110\":{\"denominacion\":\"LABORATORIO GENERAL - SCZ\",\"descripcion\":\"LAB. GRAL - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050111\":{\"denominacion\":\"SUELDOS ADMINISTRATIVOS POSTGRADO - SCZ\",\"descripcion\":\"S. ADM. POSTG - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050112\":{\"denominacion\":\"NO APLICABLES A CARRERAS - SCZ\",\"descripcion\":\"NO APLICA CAR - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050113\":{\"denominacion\":\"DISTRIBUIBLE POSTGRADO - SCZ\",\"descripcion\":\"DIST. POSTGR - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050114\":{\"denominacion\":\"ACTIVOS ACADÉMICOS DE USO GENERAL - SCZ\",\"descripcion\":\"ACT ACAD GRAL - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050115\":{\"denominacion\":\"DEPÓSITO TRANSITORIO - SCZ\",\"descripcion\":\"DEP. TRANSITO - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050116\":{\"denominacion\":\"DIRECTORES Y COORDINADORES CARR - SCZ\",\"descripcion\":\"DIREC Y COORD - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050117\":{\"denominacion\":\"SOUVENIRS BOUTIQUE - SCZ\",\"descripcion\":\"SOUVENIR BOUTQ - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050118\":{\"denominacion\":\"PROYECTO HELVETAS - SCZ\",\"descripcion\":\"PRO. HELVETAS - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"10050119\":{\"denominacion\":\"PROYECTO CHORRILLOS - SCZ\",\"descripcion\":\"PRO CHORRILLOS - SCZ\",\"departamento\":\"SANTA CRUZ\",\"area_codigo\":\"A001\",\"bloqueado\":false},\"100502\":{\"denominacion\":\"CIENCIAS DE LA SALUD - SCZ\",\"descripcion\":\"CS.SALUD - SCZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"\",\"bloqueado\":false},\"10050201\":{\"denominacion\":\"MEDICINA - SCZ\",\"descripcion\":\"MED - SCZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050202\":{\"denominacion\":\"N/ANTOLOGÍA - SCZ\",\"descripcion\":\"ODO - SCZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050203\":{\"denominacion\":\"BIOQUÍMICA Y FARMACIA - SCZ\",\"descripcion\":\"BYF - SCZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050204\":{\"denominacion\":\"FISIOTERAPIA Y KINESIOLOGÍA - SCZ\",\"descripcion\":\"LFK - SCZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050205\":{\"denominacion\":\"N/AERMERÍA CLINICO QUIRURGICA - SCZ\",\"descripcion\":\"ECQ - SCZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050206\":{\"denominacion\":\"NUTRICIÓN Y DIETÉTICA - SCZ\",\"descripcion\":\"LND - SCZ\",\"departamento\":\"CS. DE SALUD\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100503\":{\"denominacion\":\"CS. EMPRESARIALES Y SOCIALES - SCZ\",\"descripcion\":\"CS.EMP Y SOC - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"\",\"bloqueado\":false},\"10050301\":{\"denominacion\":\"ING. COMERCIAL - SCZ\",\"descripcion\":\"ICO - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050302\":{\"denominacion\":\"ING. COMERCIO INTERNACIONAL - SCZ\",\"descripcion\":\"ICT - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050303\":{\"denominacion\":\"DERECHO Y CIENCIAS JURÍDICAS - SCZ\",\"descripcion\":\"LDE - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050304\":{\"denominacion\":\"N/AUNICACIÓN Y M. DIGITALES - SCZ\",\"descripcion\":\"LCM - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050305\":{\"denominacion\":\"ADMINISTRACIÓN DE EMPRESAS - SCZ\",\"descripcion\":\"LAE - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050306\":{\"denominacion\":\"PSICOLOGÍA - SCZ\",\"descripcion\":\"LPS - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050307\":{\"denominacion\":\"N/ATADURÍA PÚBLICA - SCZ\",\"descripcion\":\"LCN - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050308\":{\"denominacion\":\"N/A. FINANCIERA Y RIESGOS - SCZ\",\"descripcion\":\"IFR - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050309\":{\"denominacion\":\"ING. CIENCIA DATOS E INT. NEGOCIOS - SCZ\",\"descripcion\":\"LCD - SCZ\",\"departamento\":\"CS EMP Y SOC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100504\":{\"denominacion\":\"INFORMÁTICA Y ELECTRÓNICA - SCZ\",\"descripcion\":\"INF Y ELEC - SCZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"\",\"bloqueado\":false},\"10050401\":{\"denominacion\":\"ING. BIOMÉDICA - SCZ\",\"descripcion\":\"IBI - SCZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050402\":{\"denominacion\":\"ING. ELECTRÓNICA - SCZ\",\"descripcion\":\"IEL - SCZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050403\":{\"denominacion\":\"ING. ELECTRO Y DE SISTEMAS - SCZ\",\"descripcion\":\"IES - SCZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050404\":{\"denominacion\":\"N/A. TELECOMUNICACIONES - SCZ\",\"descripcion\":\"IET - SCZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050405\":{\"denominacion\":\"ING. SISTEMAS INFORMÁTICOS - SCZ\",\"descripcion\":\"ISI - SCZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050406\":{\"denominacion\":\"N/A. SUP. VIDEO JUEGOS - SCZ\",\"descripcion\":\"TSV - SCZ\",\"departamento\":\"INF Y ELEC\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"100505\":{\"denominacion\":\"ARQUITECTURA Y TURISMO - SCZ\",\"descripcion\":\"ARQ Y TURIS - SCZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"\",\"bloqueado\":false},\"10050501\":{\"denominacion\":\"ARQUITECTURA Y URBANISMO - SCZ\",\"descripcion\":\"ARQ - SCZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050502\":{\"denominacion\":\"TURISMO Y HOTELERÍA - SCZ\",\"descripcion\":\"LTH - SCZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050503\":{\"denominacion\":\"GASTRONOMÍA - SCZ\",\"descripcion\":\"LGS - SCZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050504\":{\"denominacion\":\"N/A. INTERIORES Y PAISAJIS - SCZ\",\"descripcion\":\"DYP - SCZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050505\":{\"denominacion\":\"DISEÑO GRÁFICO Y COMUNIC. VISUAL - SCZ\",\"descripcion\":\"LDG - SCZ\",\"departamento\":\"ARQUI Y TUR\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100506\":{\"denominacion\":\"TECNOLOGÍA - SCZ\",\"descripcion\":\"TECNO - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"\",\"bloqueado\":false},\"10050601\":{\"denominacion\":\"N/A. PETROQUÍMICA - SCZ\",\"descripcion\":\"IPQ - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050602\":{\"denominacion\":\"ING. CIVIL - SCZ\",\"descripcion\":\"ICI - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050603\":{\"denominacion\":\"ING. PETRÓLEO, GAS Y ENERG. - SCZ\",\"descripcion\":\"IPG - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050604\":{\"denominacion\":\"N/A. INDUSTRIAS ALIMENTARIAS - SCZ\",\"descripcion\":\"IIA - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":true},\"10050605\":{\"denominacion\":\"ING. AERONÁUTICA - SCZ\",\"descripcion\":\"IAE - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050606\":{\"denominacion\":\"ING. ELECTROMECÁNICA - SCZ\",\"descripcion\":\"IEL - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050607\":{\"denominacion\":\"ING. MECÁNICA Y AUTOM. INDUSTRIAL - SCZ\",\"descripcion\":\"IMT - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050608\":{\"denominacion\":\"ING. INDUSTRIAL Y SISTEMAS - SCZ\",\"descripcion\":\"IIS - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050609\":{\"denominacion\":\"ING. INDUSTRIAL - SCZ\",\"descripcion\":\"IIN - SCZ\",\"departamento\":\"TECNOLOGIA\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100507\":{\"denominacion\":\"POSTGRADO - SCZ\",\"descripcion\":\"POSTGRADO - SCZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"\",\"bloqueado\":true},\"10050701\":{\"denominacion\":\"DOCTORADO - SCZ\",\"descripcion\":\"DOCTORADO - SCZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050702\":{\"denominacion\":\"MAESTRÍA - SCZ\",\"descripcion\":\"MAESTRIA - SCZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050703\":{\"denominacion\":\"DIPLOMADO - SCZ\",\"descripcion\":\"DIPLOMADO - SCZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"10050704\":{\"denominacion\":\"CURSOS - SCZ\",\"descripcion\":\"CURSOS - SCZ\",\"departamento\":\"POSTGRADO\",\"area_codigo\":\"A002\",\"bloqueado\":false},\"100508\":{\"denominacion\":\"LABORATORIOS - SCZ\",\"descripcion\":\"LABORATORIOS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"\",\"bloqueado\":false},\"10050801\":{\"denominacion\":\"N/A DE RADIO - SCZ\",\"descripcion\":\"SET RADIO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050802\":{\"denominacion\":\"N/A DE FOTOGRAFIA - SCZ\",\"descripcion\":\"FOTOGRAFIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050803\":{\"denominacion\":\"N/AEO GENERAL - SCZ\",\"descripcion\":\"MUSEO GRAL - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050804\":{\"denominacion\":\"LAB. DE RECURSOS FISICOS - SCZ\",\"descripcion\":\"REC FISI - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050805\":{\"denominacion\":\"LAB. DE ENTRENAMIENTO - SCZ\",\"descripcion\":\"ENTRENAM - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050806\":{\"denominacion\":\"KINESIOLOGIA - SCZ\",\"descripcion\":\"KINESIO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050807\":{\"denominacion\":\"LAB. DE SIMULACIÓN - SCZ\",\"descripcion\":\"SIMULACION - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050808\":{\"denominacion\":\"N/A. DE NEUROLOGÍA - SCZ\",\"descripcion\":\"NEURO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050809\":{\"denominacion\":\"N/A. DE PSICOMOTRICIDAD - SCZ\",\"descripcion\":\"PSICOMO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050810\":{\"denominacion\":\"LAB. HISTOLOGÍA Y PATOLOGÍA - SCZ\",\"descripcion\":\"HIST Y PATO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050811\":{\"denominacion\":\"LAB. GENETICA Y EMBRIOLOGÍA - SCZ\",\"descripcion\":\"GEN Y EMBRIO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050812\":{\"denominacion\":\"LAB. QUÍMICA Y BIOQUÍMICA - SCZ\",\"descripcion\":\"QUI Y BIOQUI - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050813\":{\"denominacion\":\"N/A. DE HEMATOLOGÍA - SCZ\",\"descripcion\":\"HEMATO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050814\":{\"denominacion\":\"SALA DE BALANZAS - SCZ\",\"descripcion\":\"S.BALANZAS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050815\":{\"denominacion\":\"SALA DE EQUIPOS - SCZ\",\"descripcion\":\"S.EQUIPOS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050816\":{\"denominacion\":\"LAB. MICROBIO Y PARASITO - SCZ\",\"descripcion\":\"MICRO Y PARAS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050817\":{\"denominacion\":\"LAB. SEMIOLOGÍA Y FISIOLOGÍA - SCZ\",\"descripcion\":\"SEMI Y FISIO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050818\":{\"denominacion\":\"N/AUGIA - SCZ\",\"descripcion\":\"CIRUGIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050819\":{\"denominacion\":\"N/ANTOPEDRIATRÍA - SCZ\",\"descripcion\":\"ODONTOPEDIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050820\":{\"denominacion\":\"N/A. DE PROTESIS - SCZ\",\"descripcion\":\"PROTESIS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050821\":{\"denominacion\":\"QUIROFANO - SCZ\",\"descripcion\":\"QUIROFANO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050822\":{\"denominacion\":\"N/AA DE ESTERILIZACIÓN - SCZ\",\"descripcion\":\"S.ESTERILI - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050823\":{\"denominacion\":\"N/ANICA (A) - SCZ\",\"descripcion\":\"CLINICA A - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050824\":{\"denominacion\":\"N/ANICA (B) - SCZ\",\"descripcion\":\"CLINICA B - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050825\":{\"denominacion\":\"N/AA DE ADMICIÓN (CIRUGIA) - SCZ\",\"descripcion\":\"CIR. ADMISION - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050826\":{\"denominacion\":\"N/AA RADIOLOGÍA (PERIEPI. PANORA) - SCZ\",\"descripcion\":\"RADIOLOGIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050827\":{\"denominacion\":\"MORFOLOGÍA SALA DE ANATOMIA - SCZ\",\"descripcion\":\"MORFOLOGIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050828\":{\"denominacion\":\"OSTEOTECA - SCZ\",\"descripcion\":\"OSTEOTECA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050829\":{\"denominacion\":\"SALA DE INVESTIGACIÓN - SCZ\",\"descripcion\":\"S. INVESTI - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050830\":{\"denominacion\":\"MUSEO DE ANATOMÍA - SCZ\",\"descripcion\":\"MUSEO ATO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050831\":{\"denominacion\":\"SALA ANATOMÍA Y SIMULACIÓN - SCZ\",\"descripcion\":\"ANATO Y SIM - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050832\":{\"denominacion\":\"DEPOSITO PIEZAS ANATOMICAS - SCZ\",\"descripcion\":\"DEP PIEZ ANAT - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050833\":{\"denominacion\":\"LAB. DE PASTELERIA - SCZ\",\"descripcion\":\"PASTELERIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050834\":{\"denominacion\":\"LAB. DE COCINA - SCZ\",\"descripcion\":\"COCINA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050835\":{\"denominacion\":\"ECONOMATO - SCZ\",\"descripcion\":\"ECONOMATO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050836\":{\"denominacion\":\"LAB. DE PANADERIA - SCZ\",\"descripcion\":\"PANADERIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050837\":{\"denominacion\":\"COCTELERIA - SCZ\",\"descripcion\":\"COCTELERIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050838\":{\"denominacion\":\"N/AA DE PEDIATRÍA - SCZ\",\"descripcion\":\"S. PEDIATRIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050839\":{\"denominacion\":\"N/AA GINECO. Y OBSTETRICIA - SCZ\",\"descripcion\":\"S. GINE Y OBS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050840\":{\"denominacion\":\"SALA DE SUMINISTROS - SCZ\",\"descripcion\":\"S. SUMINISTRO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050841\":{\"denominacion\":\"N/AA DE QUIROFANO - SCZ\",\"descripcion\":\"S. QUIROFANO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050842\":{\"denominacion\":\"N/AA DE HOSPITALIZACIÓN - SCZ\",\"descripcion\":\"S. HOSPITAL - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050843\":{\"denominacion\":\"N/AA DE TERAPIA INTENSIVA - SCZ\",\"descripcion\":\"S. TER INTENS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050844\":{\"denominacion\":\"N/AA DE ATENCIÓN PRIMARIA - SCZ\",\"descripcion\":\"S. ATEN PRIM - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050845\":{\"denominacion\":\"N/AA MULTIPROPOSITO - SCZ\",\"descripcion\":\"S. MULTIPROP - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050846\":{\"denominacion\":\"TALLER DE ARQUITECTURA - SCZ\",\"descripcion\":\"T. ARQUITEC - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050847\":{\"denominacion\":\"GB. URBANISMO Y TRAFICO - SCZ\",\"descripcion\":\"G. URB Y TRAF - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050848\":{\"denominacion\":\"INCUBACIÓN - SCZ\",\"descripcion\":\"INCUBACION - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050849\":{\"denominacion\":\"SALA DE JUICIOS ORALES - SCZ\",\"descripcion\":\"S. JUICIOS OR - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050850\":{\"denominacion\":\"NEUROMARKETING - SCZ\",\"descripcion\":\"NEUROMKT - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050851\":{\"denominacion\":\"CONSERVATORIO Y SALA REUNIONES - SCZ\",\"descripcion\":\"CONSER Y REUN - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050852\":{\"denominacion\":\"INNOVACIÓN - SCZ\",\"descripcion\":\"INNOVACION - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050853\":{\"denominacion\":\"SALA DE REUNIONES - SCZ\",\"descripcion\":\"S. REUNION - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050854\":{\"denominacion\":\"CAMARA GESELL - SCZ\",\"descripcion\":\"CAM GESELL - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050855\":{\"denominacion\":\"CENTRO DE COMPUTO - SCZ\",\"descripcion\":\"CEN COMPUTO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050856\":{\"denominacion\":\"LAB. DE BURSATIL - SCZ\",\"descripcion\":\"BURSATIL - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050857\":{\"denominacion\":\"LAB. DISEÑO COMPUTARIZADO - SCZ\",\"descripcion\":\"DIS COMPUTA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050858\":{\"denominacion\":\"LAB. DE FISICA - SCZ\",\"descripcion\":\"FISICA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050859\":{\"denominacion\":\"LAB. ELECTRÓNICA DIGITAL - SCZ\",\"descripcion\":\"ELECT DIGITAL - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050860\":{\"denominacion\":\"N/A. DE ROBOTICA - SCZ\",\"descripcion\":\"ROBOTICA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050861\":{\"denominacion\":\"LAB. TELECOMUNICACIONES - SCZ\",\"descripcion\":\"L. TELECOM - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050862\":{\"denominacion\":\"LAB. ELECTRÓNICA Y BIOMÉDICA - SCZ\",\"descripcion\":\"L. ELEC Y BIO - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050863\":{\"denominacion\":\"N/A. ENERGIAS ALTERNATIVAS - SCZ\",\"descripcion\":\"ENER ALTER - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050864\":{\"denominacion\":\"N/A. LUBRICANTES Y CARBURANTES - SCZ\",\"descripcion\":\"LUBR Y CAR - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050865\":{\"denominacion\":\"N/A. SIMULACIÓN INDUS. INGENI MET - SCZ\",\"descripcion\":\"SIM INDUSTRIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050866\":{\"denominacion\":\"LAB. ANL. INSTRUMEN, GB METRO IND - SCZ\",\"descripcion\":\"A IND ING MET - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050867\":{\"denominacion\":\"N/A SEGURIDAD INDUST. Y TOPOG. - SCZ\",\"descripcion\":\"SEG IND Y TOP - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050868\":{\"denominacion\":\"N/A. DE SANITARIA - SCZ\",\"descripcion\":\"SANITARIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050869\":{\"denominacion\":\"N/A. HIDRAULICA - SCZ\",\"descripcion\":\"HIDRAULICA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050870\":{\"denominacion\":\"N/A. SUELOS, HORMIGONES Y ASFALTO - SCZ\",\"descripcion\":\"SUELOS HORMIG - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050871\":{\"denominacion\":\"N/A. RESISTENCIA MATERIALES - SCZ\",\"descripcion\":\"RESIS MATER - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050872\":{\"denominacion\":\"N/A. PROCESOS INDUSTRIALES - SCZ\",\"descripcion\":\"PROC INDUS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050873\":{\"denominacion\":\"N/A. ANALISIS ESTRUCTURAL - SCZ\",\"descripcion\":\"ANALIS ESTRUC - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050874\":{\"denominacion\":\"N/A. DE DESTILACIÓN - SCZ\",\"descripcion\":\"DESTILACION - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050875\":{\"denominacion\":\"N/A. CONDUCTUAL - SCZ\",\"descripcion\":\"CONDUCTUAL - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050876\":{\"denominacion\":\"N/A. PLANTA DE ALIMENTOS - SCZ\",\"descripcion\":\"PLANTA ALIM. - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050877\":{\"denominacion\":\"N/A. EMBUTIDOS - SCZ\",\"descripcion\":\"EMBUTIDOS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050878\":{\"denominacion\":\"N/A. AUDIO DIGITAL - SCZ\",\"descripcion\":\"AUDIO DIGITAL - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050879\":{\"denominacion\":\"N/A. SET DE TELEVISION - SCZ\",\"descripcion\":\"ST TELEVISION - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050880\":{\"denominacion\":\"LAB QUIMICA ORGANICA E INORGANICA - SCZ\",\"descripcion\":\"QMC ORG/INORG - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050881\":{\"denominacion\":\"N/A. FISICOQUIMICA - SCZ\",\"descripcion\":\"FISICOQUIMICA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050882\":{\"denominacion\":\"LAB. PARASITOLOGIA - SCZ\",\"descripcion\":\"PARASITOLOGIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050883\":{\"denominacion\":\"N/A. ELECTROTENIA - SCZ\",\"descripcion\":\"ELECTROTENIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050884\":{\"denominacion\":\"N/A. POTENCIA - SCZ\",\"descripcion\":\"POTENCIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050885\":{\"denominacion\":\"LAB. MAQUINAS ELECTRICAS - SCZ\",\"descripcion\":\"MAQ. ELECTR. - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050886\":{\"denominacion\":\"LAB. MAQUINAS TERMICAS - SCZ\",\"descripcion\":\"MAQ. TERMIC. - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050887\":{\"denominacion\":\"LAB. PROCESOS - SCZ\",\"descripcion\":\"PROCESOS - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050888\":{\"denominacion\":\"LAB. ANALISIS SENSORIAL - SCZ\",\"descripcion\":\"ANALIS SENSOR - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050889\":{\"denominacion\":\"N/A. OBRAS HIDRAULICAS - SCZ\",\"descripcion\":\"OBRAS HIDRAU - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050890\":{\"denominacion\":\"N/A. AUTOMOTORES - SCZ\",\"descripcion\":\"AUTOMOTORES - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":true},\"10050891\":{\"denominacion\":\"TALLER DE METALMECANICA - SCZ\",\"descripcion\":\"METALMECANICA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050892\":{\"denominacion\":\"BIBLIOTECA - SCZ\",\"descripcion\":\"BIBLIOTECA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050893\":{\"denominacion\":\"LAB. HABILIDADES QUIRURGICAS - SCZ\",\"descripcion\":\"HAB QUIRURG - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050894\":{\"denominacion\":\"LAB. HABILIDADES COMUNITARIAS - SCZ\",\"descripcion\":\"HAB COMUNITA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050895\":{\"denominacion\":\"LAB. HABILIDADES ENFERMERIA - SCZ\",\"descripcion\":\"HAB ENFERMER - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050896\":{\"denominacion\":\"GAB. SEGURIDAD E HIGIENE INDUST - SCZ\",\"descripcion\":\"SEG E HIG IND - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050897\":{\"denominacion\":\"LAB. INDUSTRIAS PETROQUIMICAS - SCZ\",\"descripcion\":\"IND PETROQUI - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050898\":{\"denominacion\":\"LAB. EVALUACIÓN NUTRICIONAL - SCZ\",\"descripcion\":\"EV NUTRICIONAL - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050899\":{\"denominacion\":\"SALA DE ESPECIALIDADES - SCZ\",\"descripcion\":\"S ESPECIALIDAD - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050900\":{\"denominacion\":\"FABLAB - SCZ\",\"descripcion\":\"FABLAB - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050901\":{\"denominacion\":\"NUCLEO ASESORAMIENTO EMPRESARIAL - SCZ\",\"descripcion\":\"NAE - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050902\":{\"denominacion\":\"LAB. SERIGRAFÍA - SCZ\",\"descripcion\":\"SERIGRAFIA - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false},\"10050903\":{\"denominacion\":\"LAB. KINCO - SCZ\",\"descripcion\":\"KINCO LAB - SCZ\",\"departamento\":\"LABORATORIOS\",\"area_codigo\":\"A003\",\"bloqueado\":false}}")

# ============================================================
# EJECUCIÓN SECUENCIAL NATIVA DE GOOGLE COLAB (sin ipywidgets)
# ============================================================
import time
from pathlib import Path

from IPython.display import HTML, FileLink, display


def _safe_name(text):
    return "_".join(str(text).replace("/", "-").split())


def _identify_inputs(paths):
    siat_path = None
    sap_path = None
    diagnostics = []
    for path in paths:
        sheets = set(pd.ExcelFile(path).sheet_names)
        diagnostics.append(f"{path.name}: {', '.join(sorted(sheets))}")
        if "SIAT" in sheets:
            if siat_path is not None:
                raise ValueError("Se detectaron dos archivos con pestaña SIAT.")
            siat_path = path
        if "SAP Document Export" in sheets:
            if sap_path is not None:
                raise ValueError("Se detectaron dos archivos con pestaña SAP Document Export.")
            sap_path = path
    if siat_path is None or sap_path is None:
        raise ValueError(
            "No pude identificar ambos archivos. Uno debe contener la pestaña SIAT "
            "y el otro la pestaña SAP Document Export.\n" + "\n".join(diagnostics)
        )
    return siat_path, sap_path


PROGRESS_STAGES = {
    "recibidos":     (10,  "Archivos recibidos"),
    "identificados": (25,  "Archivos identificados"),
    "leyendo":       (40,  "Leyendo información"),
    "cruzando":      (60,  "Cruzando facturas"),
    "validando":     (75,  "Validando anulaciones y reversiones"),
    "generando":     (90,  "Generando reporte"),
    "listo":         (100, "Reporte terminado"),
}

_progress_handle = None


def _progress_bar_html(percent: int, title: str, detail: str, color: str = "#2F75B5") -> HTML:
    percent = max(0, min(100, int(percent)))
    return HTML(f"""
    <div style="font-family:Arial, sans-serif; max-width:720px; margin-top:6px">
      <div style="display:flex; justify-content:space-between; font-size:13px; margin-bottom:4px">
        <b style="color:#17365D">{title}</b>
        <span style="color:#595959">{percent}%</span>
      </div>
      <div style="background:#E7ECF3; border-radius:8px; height:20px; overflow:hidden">
        <div style="background:{color}; width:{percent}%; height:100%; transition:width .35s ease"></div>
      </div>
      <div style="margin-top:6px; font-size:13px; color:#404040">{detail}</div>
    </div>
    """)


def show_progress(stage: str, detail: str = "", color: str = "#2F75B5") -> None:
    global _progress_handle
    percent, title = PROGRESS_STAGES[stage]
    html = _progress_bar_html(percent, title, detail or title, color=color)
    if _progress_handle is None:
        _progress_handle = display(html, display_id=True)
    else:
        _progress_handle.update(html)
    time.sleep(0.05)


def show_error(detail: str) -> None:
    global _progress_handle
    html = _progress_bar_html(100, "Proceso detenido", detail, color="#C00000")
    if _progress_handle is None:
        _progress_handle = display(html, display_id=True)
    else:
        _progress_handle.update(html)
    print("\n✗ " + detail)


display(HTML(
    "<div style='font-family:Arial, sans-serif; background:linear-gradient(90deg,#17365D,#2F75B5);"
    "padding:18px 24px; border-radius:12px; color:#FFFFFF; margin-bottom:14px'>"
    "<div style='font-size:23px; font-weight:bold'>🚀 GENERAR DETERMINACIÓN IVA–IT</div>"
    "<div style='font-size:13px; opacity:.9; margin-top:4px'>"
    "Automatizador UNIVALLE &middot; Cruce SIAT (NetValle) &times; Mayor SAP</div></div>"
))

work_dir = Path("/content/automatizador_tributario")
work_dir.mkdir(parents=True, exist_ok=True)
for old_file in work_dir.glob("*"):
    if old_file.is_file():
        old_file.unlink()

report_path = None
try:
    print("Selecciona juntos los 2 archivos .xlsx: el CRUCE OFICIAL (pestaña SIAT) "
          "y el mayor SAP (pestaña 'SAP Document Export').\n")
    try:
        from google.colab import files as colab_files
    except ImportError as exc:
        raise RuntimeError(
            "Este panel usa el selector nativo de Google Colab (google.colab.files). "
            "Ábrelo y ejecútalo dentro de Google Colab."
        ) from exc

    uploaded = colab_files.upload()

    if len(uploaded) != 2:
        raise ValueError(
            f"Se recibieron {len(uploaded)} archivo(s). Debes seleccionar exactamente 2 "
            "archivos .xlsx: el CRUCE OFICIAL y el mayor SAP."
        )
    non_xlsx = [name for name in uploaded if not name.lower().endswith(".xlsx")]
    if non_xlsx:
        raise ValueError(f"Estos archivos no son .xlsx: {', '.join(non_xlsx)}")

    print(f"✓ {len(uploaded)} archivos recibidos\n")
    show_progress("recibidos", "Guardando los archivos recibidos...")

    saved_paths = []
    for name, content in uploaded.items():
        path = work_dir / name
        path.write_bytes(content)
        saved_paths.append(path)

    show_progress("identificados", "Identificando SIAT y SAP por su estructura...")
    siat_path, sap_path = _identify_inputs(saved_paths)
    print(f"✓ SIAT identificado: {siat_path.name}")
    print(f"✓ SAP identificado: {sap_path.name}\n")

    show_progress("leyendo", "Leyendo el Libro de Ventas SIAT y el mayor SAP...")
    siat_frame, sap_frame = load_inputs(siat_path, sap_path)

    sap_accounts = set(sap_frame["Cuenta de mayor"].map(normalize_account))
    if not any(account.startswith("4") for account in sap_accounts):
        raise ValueError(
            "El mayor SAP no contiene cuentas de ingreso (cuentas que comienzan con '4')."
        )

    show_progress(
        "cruzando",
        "Cruzando facturas por número, fecha y caja; procesando anulaciones y reversiones...",
    )
    result = process_data(siat_frame, sap_frame, cebe_map=CEBE_MAP)

    show_progress("validando", "Validando que el IVA y el IT cuadren contra el mayor SAP...")
    if abs(result["assignment_difference"]) >= 0.005:
        raise ValueError(
            "El IT no cerró contra el mayor SAP: diferencia Bs "
            f"{result['assignment_difference']:,.2f}. Se detuvo la exportación; revisa el "
            "archivo de origen antes de reintentar."
        )

    show_progress(
        "generando",
        "Construyendo el Excel final (Resumen, Libro de ventas, Respaldo IT y Detalle CeBe)...",
    )
    period_name = _safe_name(result["period"]["label"])
    report_path = work_dir / f"Determinacion_IVA_IT_{period_name}.xlsx"
    build_report(result, report_path)

    show_progress(
        "listo", f"Reporte de {result['period']['label']} generado correctamente.", color="#2E7D32"
    )

    print("\nREPORTE GENERADO CORRECTAMENTE")
    print(f"Periodo: {result['period']['label']}")
    print(f"Registros SIAT: {result['input_rows']['siat']:,}")
    print(f"Partidas SAP: {result['input_rows']['sap']:,}")
    print(f"IT mayor SAP: Bs {result['it_source_total']:,.2f}")
    print(f"IT asignado a facturas: Bs {result['it_assigned_total']:,.2f}")
    print(f"Diferencia IT: Bs {result['assignment_difference']:,.2f}")
    print(f"Grupos de reversión/reclasificación neto cero: {result['zero_groups']:,}")
    print(
        "Excepciones para revisión: "
        f"{len(result['exceptions']) + len(result['unknown_income_accounts']):,}"
    )

except Exception as exc:
    show_error(str(exc))
    report_path = None

if report_path is not None:
    print("\nDescargando el Excel...")
    try:
        colab_files.download(str(report_path))
    except Exception:
        print("La descarga automática no se pudo iniciar. Usa este enlace manual:")
        display(FileLink(str(report_path), result_html_prefix="", result_html_suffix=""))
